# LINet3 Training + Diagnostics on SUN RGB-D

**Complete training pipeline with gradient health monitoring, stream contribution analysis, and internal CNN visualization**

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime -> Change runtime type -> Hardware accelerator: GPU -> GPU type: A100
- [ ] **Mount Google Drive:** Your code and dataset will be stored on Drive
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_15_train_test.tar.gz` (train + test splits)

---

## What This Notebook Does:

**Training with Full Diagnostics:**
1. Train LINet3 (2-stream: RGB + HHA) on SUN RGB-D 19-category
2. Gradient health monitoring (vanishing/exploding/oscillating detection)
3. Per-stream training loss decomposition
4. Integration weight evolution tracking

**Post-Training Visualization Suite:**
5. Feature map visualization (full model, per-stream, ablation)
6. Stream contribution decomposition (what each stream contributes to each neuron)
7. Stream-decomposed Grad-CAM (where each stream focuses attention)
8. Integration weight analysis (learned fusion priorities per layer)
9. Stream redundancy analysis (are streams learning the same thing?)
10. Per-class stream dominance (which scenes rely on RGB vs HHA?)
11. Misclassification analysis with Grad-CAM comparison
12. Train vs test activation divergence + BN stats reset experiment

---

## About LINet3:

**LINet3** (Linear Integration Network v3) is an N-stream ResNet where fusion happens **inside each convolution neuron**:
- Per-stream independent convolution kernels (full spatial filters)
- Learned 1x1 integration weights that combine stream outputs at every layer
- Integrated pathway carries the fused representation forward

This allows the network to learn **layer-specific, spatially-aware integration strategies**.

## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU Memory: 94.97 GB

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition



In [2]:
# Detailed GPU info
!nvidia-smi

Thu Apr 23 19:17:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   37C    P0             64W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted successfully!

Drive contents:
total 3118192
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

REPOSITORY SETUP
Repo already exists: /content/Multi-Stream-Neural-Networks
Already up to date.

Working directory: /content/Multi-Stream-Neural-Networks
total 88
drwxr-xr-x 12 root root  4096 Apr 23 18:51 .
drwxr-xr-x  1 root root  4096 Apr 23 18:51 ..
drwxr-xr-x  5 root root  4096 Apr 23 18:51 configs
drwxr-xr-x  2 root root  4096 Apr 23 18:51 data
drwxr-xr-x  5 root root  4096 Apr 23 18:51 docs
drwxr-xr-x  3 root root  4096 Apr 23 18:51 experiments
drwxr-xr-x  9 root root  4096 Apr 23 19:17 .git
-rw-r--r--  1 root root   732 Apr 23 18:51 .gitattributes
drwxr-xr-x  3 root root  4096 Apr 23 18:51 .github
-rw-r--r--  1 root root   847 Apr 23 18:51 .gitignore
-rw-r--r--  1 root root  1084 Apr 23 18:51 LICENSE
drwxr-xr-x  2 root root  4096 Apr 23 18:51 notebooks
-rw-r--r--  1 root root   198 Apr 23 18:51 pytest.ini
-rw-r--r--  1 root root  6920 Apr 23 18:51 README.md
-rw-r--r--  1 root root   126 Apr 23 18:51 requirements.txt
drwxr-xr-x  2 root root  4096 Apr 23 18:51 scripts
-rw-r--r-- 

## 4. Install Dependencies

In [5]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

Installing dependencies...
All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   kornia: 0.8.2
   thop: 0.1.1


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed (train + test splits, RGB + HHA)

In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_hha"  # Extracted location

print("=" * 60)
print("SUN RGB-D 15-CATEGORY DATASET SETUP (TRAIN + TEST)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")

    # Verify structure
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    # Copy compressed file with progress
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} /dev/shm/sunrgbd_19_hha.tar.gz

    # Extract to local disk
    print(f"\nExtracting dataset to local disk...")
    !tar -xzf /dev/shm/sunrgbd_19_hha.tar.gz -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    # Remove tar file to save space
    !rm /dev/shm/sunrgbd_19_hha.tar.gz

    print(f"\nDataset extracted to local disk")

    # Verify extraction
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

SUN RGB-D 15-CATEGORY DATASET SETUP (TRAIN + TEST)
Dataset already on local disk: /dev/shm/sunrgbd_19_traintest

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Import LINet3

In [7]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and dataloaders
print("\nImporting LiNet, dataloaders, and visualization tools...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.common.model_helpers import load_pretrained_backbone
from src.models.linear_integration.li_net3.conv import LIBatchNorm2d
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders
from src.training.augmentation_config import AugmentationConfig

# Import visualization suite
from src.utils.visualization import (
    FeatureMapVisualizer,
    StreamContributionVisualizer,
    StreamGradCAM,
    IntegrationWeightVisualizer,
    find_misclassified,
    compare_samples,
    StreamRedundancyAnalyzer,
    PerClassDominanceAnalyzer,
    ActivationDivergenceAnalyzer,
    IntegrationWeightEvolutionVisualizer,
    reset_bn_stats,
)

print("All imports successful!")

Project structure:
total 52
drwxr-xr-x 12 root root 4096 Apr 23 18:52 .
drwxr-xr-x  8 root root 4096 Apr 23 18:52 ..
drwxr-xr-x  3 root root 4096 Apr 23 18:52 abstracts
drwxr-xr-x  3 root root 4096 Apr 23 18:52 common
drwxr-xr-x  3 root root 4096 Apr 23 18:52 core
drwxr-xr-x  2 root root 4096 Apr 23 18:51 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Apr 23 18:51 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Apr 23 18:51 direct_mixing_conv
-rw-r--r--  1 root root 1076 Apr 23 18:51 __init__.py
drwxr-xr-x  5 root root 4096 Apr 23 18:52 linear_integration
drwxr-xr-x  3 root root 4096 Apr 23 18:52 multi_channel
drwxr-xr-x  2 root root 4096 Apr 23 18:52 __pycache__
drwxr-xr-x  2 root root 4096 Apr 23 18:51 utils

Importing LiNet, dataloaders, and visualization tools...
All imports successful!


In [8]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 152
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

Seed: 152, Deterministic: False


## 7. Configuration

All hyperparameters and settings in one place. Modify these before running.

In [9]:
from src.training.augmentation_config import AugmentationConfig

# ======================== DATASET ========================
DATASET_CONFIG = {
    'data_root': LOCAL_DATASET_PATH,
    'batch_size': 64,
    'num_workers': 2,
    'num_classes': 19,
    'seed': SEED
}

AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.04,
    rgb_aug_mag=1.24,
    depth_aug_prob=1.00,
    depth_aug_mag=1.29,
)

# ======================== MODEL ========================
MODEL_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 19,
    'stream_input_channels': [3, 3],  # RGB=3, HHA=3
    'width_multiplier': 0.75,
    'dropout_p': 0.69,
    'device': 'cuda',
    'use_amp': True
}

STREAM_LABELS = {0: 'RGB', 1: 'HHA'}

# ======================== PRETRAINED WEIGHTS (Optional) ========================
# Set LOAD_PRETRAINED = True to initialize from ScanNet pretrained backbone.
# The fc head is skipped automatically since num_classes differs.
LOAD_PRETRAINED = True
PRETRAINED_WEIGHTS_PATH = "/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt"  # TODO: set path
FREEZE_BACKBONE_EPOCHS = 3  # Set > 0 to freeze backbone for N warmup epochs (only used when LOAD_PRETRAINED = True)
FREEZE_BACKBONE_LR = 1e-3     # Learning rate for classifier warmup phase

# ======================== OPTIMIZER ========================
STREAM_SPECIFIC_CONFIG = {
    'stream_lrs': 2.048e-04,        # [RGB, HHA]
    # 'shared_lr': 1.948e-04,
    'stream_weight_decays': 1.255e-03,  # [RGB, HHA]
    # 'integration_weight_decay': 1.944e-05,
    'stem_lr_multiplier': 3.10,  # >1.0 to boost stem LR (changes eta_min to 6 values)
}

SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 25,
    # 's1_eta': 1.944e-05,
    # 's2_eta': 1.944e-05,
    'eta_min': 2.414e-06,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2
}

# ======================== TRAINING ========================
TRAIN_CONFIG = {
    'epochs': 30,
    'grad_clip_norm': 0.90,
    'early_stopping': False,
    'restore_best_weights': False,
    'stream_monitoring': False,
    'modality_dropout': True,
    'modality_dropout_start':0,
    'modality_dropout_ramp':0,
    'modality_dropout_rate': 0.49,
    'label_smoothing': 0.06,
    # Gradient health monitoring
    'gradient_monitoring': False,
    'gradient_log_freq': 0,  # Last batch per epoch
    # Integration weight tracking
    'track_integration_weights': False,
    'integration_snapshot_freq': 10,
    'monitor': 'val_mca',
}

# Print summary
print('All configs defined.')
print(f'  Dataset: {DATASET_CONFIG["data_root"]}')
print(f'  Model: LINet3-{MODEL_CONFIG["architecture"]} ({len(MODEL_CONFIG["stream_input_channels"])}-stream)')
print(f'  Streams: {STREAM_LABELS}')
print(f'  Epochs: {TRAIN_CONFIG["epochs"]}, Grad clip: {TRAIN_CONFIG["grad_clip_norm"]}')
print(f'  Gradient monitoring: {TRAIN_CONFIG["gradient_monitoring"]}')
print(f'  Integration weight tracking: {TRAIN_CONFIG["track_integration_weights"]}')

All configs defined.
  Dataset: /dev/shm/sunrgbd_19_traintest
  Model: LINet3-resnet18 (2-stream)
  Streams: {0: 'RGB', 1: 'Depth'}
  Epochs: 30, Grad clip: 0.9
  Gradient monitoring: False
  Integration weight tracking: False


## 8. Load Dataset

In [10]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"    {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"      {modality}/ - {len(list(mod_dir.glob('*.png')))} images")
        print(f"      labels.txt")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip() for line in f]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

DATASET STRUCTURE VERIFICATION

Directory structure:
  /dev/shm/sunrgbd_19_traintest/
    train/
      labels.txt
    test/
      labels.txt

Classes (19):
  0: 0: bathroom
  1: 1: bedroom
  2: 2: classroom
  3: 3: computer_room
  4: 4: conference_room
  5: 5: corridor
  6: 6: dining_area
  7: 7: dining_room
  8: 8: discussion_area
  9: 9: furniture_store
  10: 10: home_office
  11: 11: kitchen
  12: 12: lab
  13: 13: lecture_theatre
  14: 14: library
  15: 15: living_room
  16: 16: office
  17: 17: rest_space
  18: 18: study_space



In [11]:
print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN + TEST)")
print("=" * 60)

print(f"\nLoading dataset from: {DATASET_CONFIG['data_root']}")

# Create dataloaders (val_loader will be None since no val/ directory)
train_loader, val_loader, test_loader = get_sunrgbd_dataloaders(
    data_root=DATASET_CONFIG['data_root'],
    batch_size=DATASET_CONFIG['batch_size'],
    num_workers=DATASET_CONFIG['num_workers'],
    seed=DATASET_CONFIG['seed'],
    **AUGMENTATION_CONFIG.to_dict(),
    stratified=True,
    normalize=True,
    use_hha=True,
)

print(f"\nDataset loaded!")
print(f"  Train: {len(train_loader.dataset)} samples ({len(train_loader)} batches)")
print(f"  Test: {len(test_loader.dataset)} samples ({len(test_loader)} batches)")
print(f"  Val: {'None (no val split)' if val_loader is None else f'{len(val_loader.dataset)} samples'}")

# Test loading a batch
# rgb_batch, depth_batch, label_batch = next(iter(train_loader))
# print(f"\nBatch shapes: RGB={rgb_batch.shape}, HHA={depth_batch.shape}, Labels={label_batch.shape}")

print("\n" + "=" * 60)

LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN + TEST)

Loading dataset from: /dev/shm/sunrgbd_19_traintest
Loaded SUN RGB-D train: 4845 samples, 19 classes (tensors, mmap)

Augmentation scaling applied:
  RGB:   prob=1.04, mag=1.24
  Depth: prob=1.00, mag=1.29
  Computed values:
    [Sync]  Flip prob: 0.50 -> 0.510
    [RGB]   ColorJitter prob: 0.43 -> 0.447
    [RGB]   Brightness: ±0.37 -> ±0.459
    [RGB]   Blur prob: 0.25 -> 0.260
    [RGB]   Grayscale prob: 0.17 -> 0.177
    [RGB]   Erasing prob: 0.17 -> 0.177
    [Depth] Aug prob: 0.50 -> 0.500
    [Depth] Brightness: ±0.25 -> ±0.323
    [Depth] Noise std: 0.059 -> 0.076
    [Depth] Erasing prob: 0.10 -> 0.100
Loaded SUN RGB-D test: 4659 samples, 19 classes (tensors, mmap)

Stratified sampling enabled (training only):
  Train class imbalance: 14.6x
  Each training batch will have balanced class representation

DataLoader Info:
  Train batches: 76
  Val batches: N/A (no val split)
  Test batches: 73
  Batch size: 64
  Stratified: Tr

## 9. Create Model

In [12]:
from src.models.linear_integration.li_net3 import li_resnet18
from thop import profile


print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

model = li_resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    stream_input_channels=MODEL_CONFIG['stream_input_channels'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    dropout_p=MODEL_CONFIG['dropout_p'],
    device=MODEL_CONFIG['device'],
    use_amp=MODEL_CONFIG['use_amp']
)

# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# # GFLOPs calculation — register custom handlers for LI modules
# from src.models.linear_integration.li_net3.conv import LIConv2d, LIBatchNorm2d
# from src.models.linear_integration.li_net3.container import LIReLU
# from src.models.linear_integration.li_net3.pooling import LIMaxPool2d, LIAdaptiveAvgPool2d

# def _liconv2d_flops(module, input, output):
#     stream_outs, integrated_out = output
#     total = 0
#     for w, s_out in zip(module.stream_weights, stream_outs):
#         batch, out_c, out_h, out_w = s_out.shape
#         kernel_ops = w.shape[1] * w.shape[2] * w.shape[3]
#         total += batch * out_c * out_h * out_w * kernel_ops
#     if module.integrated_weight.shape[1] > 0:
#         batch, out_c, out_h, out_w = integrated_out.shape
#         kernel_ops = module.integrated_weight.shape[1]
#         total += batch * out_c * out_h * out_w * kernel_ops
#     for iw in module.integration_from_streams:
#         batch, out_c, out_h, out_w = integrated_out.shape
#         kernel_ops = iw.shape[1]
#         total += batch * out_c * out_h * out_w * kernel_ops
#     module.total_ops += torch.DoubleTensor([total])

# def _libn_flops(module, input, output):
#     stream_outs, integrated_out = output
#     total = sum(s.numel() for s in stream_outs) + integrated_out.numel()
#     module.total_ops += torch.DoubleTensor([total * 4])

# def _lirelu_flops(module, input, output):
#     stream_outs, integrated_out = output
#     module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

# def _lipool_flops(module, input, output):
#     stream_outs, integrated_out = output
#     module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

# custom_ops = {
#     LIConv2d: _liconv2d_flops,
#     LIBatchNorm2d: _libn_flops,
#     LIReLU: _lirelu_flops,
#     LIMaxPool2d: _lipool_flops,
#     LIAdaptiveAvgPool2d: _lipool_flops,
# }

# dummy_streams = [torch.randn(1, ch, 224, 224).to(MODEL_CONFIG['device']) for ch in MODEL_CONFIG['stream_input_channels']]
# li_flops, _ = profile(model, inputs=(dummy_streams,), custom_ops=custom_ops, verbose=False)
# del dummy_streams

# print(f"\nLINet3-{MODEL_CONFIG['architecture'].upper()} created")
# print(f"  Total parameters: {total_params:,}")
# print(f"  GFLOPs: {li_flops / 1e9:.3f}")
# print(f"  Model size: {total_params * 4 / 1024**2:.2f} MB (FP32)")
# print(f"  Streams: {STREAM_LABELS}")
# print(f"  AMP: {MODEL_CONFIG['use_amp']}")

# Load pretrained backbone weights (optional)
if LOAD_PRETRAINED:
    print(f"\nLoading pretrained backbone from: {PRETRAINED_WEIGHTS_PATH}")
    transfer_info = load_pretrained_backbone(model, PRETRAINED_WEIGHTS_PATH)
    print(f"  Loaded:  {len(transfer_info['loaded'])} keys (backbone)")
    print(f"  Skipped: {len(transfer_info['skipped'])} keys (classifier head)")
else:
    print("\nTraining from scratch (LOAD_PRETRAINED = False)")

print("\n" + "=" * 60)

MODEL CREATION
✅ Enabled Automatic Mixed Precision (AMP) training on cuda

Loading pretrained backbone from: /content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt
Loaded pretrained backbone from: /content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt
  Loaded:  360 keys
  Skipped: 2 keys (shape mismatch or absent)
  Skipped keys: ['fc.weight', 'fc.bias']
  Loaded: 360 keys (backbone)
  Skipped: 2 keys (classifier head)



## 10. Compile Model (Optimizer + Scheduler)

In [13]:
import os
from datetime import datetime
from pathlib import Path

# Create checkpoint directory on Google Drive (persistent storage)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/run_{timestamp}"

Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {checkpoint_dir}")

Checkpoint directory: /content/drive/MyDrive/linet_checkpoints/run_20260423_191732


In [14]:
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler

print("=" * 60)
print("MODEL COMPILATION")
print("=" * 60)

# Add checkpoint-dependent paths to TRAIN_CONFIG
TRAIN_CONFIG['save_path'] = f"{checkpoint_dir}/best_model.pt"
TRAIN_CONFIG['integration_snapshot_path'] = f"{checkpoint_dir}/integration_snapshots"


def _create_main_optimizer_and_scheduler():
    """Create the main optimizer + scheduler (used after optional warmup)."""
    opt = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[STREAM_SPECIFIC_CONFIG['stream_lrs'], STREAM_SPECIFIC_CONFIG['stream_lrs']],
        stream_weight_decays=[STREAM_SPECIFIC_CONFIG['stream_weight_decays'], STREAM_SPECIFIC_CONFIG['stream_weight_decays']],
        shared_lr=STREAM_SPECIFIC_CONFIG['stream_lrs'],
        integration_weight_decay=STREAM_SPECIFIC_CONFIG['stream_weight_decays'],
        stem_lr_multiplier=STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'],
    )
    sched = setup_scheduler(
        opt,
        scheduler_type=SCHEDULER_CONFIG['scheduler_type'],
        train_loader_len=len(train_loader),
        t_max=SCHEDULER_CONFIG['t_max'],
        eta_min=(
            ([SCHEDULER_CONFIG['eta_min'] * STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'],
              SCHEDULER_CONFIG['eta_min'] * STREAM_SPECIFIC_CONFIG['stem_lr_multiplier']]
             if STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'] != 1.0 else []) +
            [SCHEDULER_CONFIG['eta_min'], SCHEDULER_CONFIG['eta_min'],
             SCHEDULER_CONFIG['eta_min'], SCHEDULER_CONFIG['eta_min']]
        ),
        warmup_epochs=SCHEDULER_CONFIG['warmup_epochs'],
        warmup_start_factor=SCHEDULER_CONFIG['warmup_start_factor']
    )
    return opt, sched


# --- Optional backbone freeze warmup ---
freeze_epochs = FREEZE_BACKBONE_EPOCHS if LOAD_PRETRAINED else 0
warmup_history = None

if freeze_epochs > 0:
    print(f"\nBACKBONE FREEZE WARMUP ({freeze_epochs} epochs)")
    print("-" * 40)

    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    # trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    # frozen_count = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    # print(f"  Frozen: {frozen_count:,} params | Trainable: {trainable_count:,} params")

    fc_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=FREEZE_BACKBONE_LR,
    )
    model.compile(
        optimizer=fc_optimizer,
        scheduler=None,
        loss='cross_entropy',
        label_smoothing=TRAIN_CONFIG['label_smoothing'],
        gpu_augmentation=False,
        **AUGMENTATION_CONFIG.to_dict(),
    )

    warmup_history = model.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=freeze_epochs,
        verbose=True,
        # grad_clip_norm=TRAIN_CONFIG['grad_clip_norm'],
        stream_monitoring=False,
        modality_dropout=False,
        gradient_monitoring=False,
        track_integration_weights=False,
    )

    for param in model.parameters():
        param.requires_grad = True
    print(f"\n  All parameters unfrozen.")

elif LOAD_PRETRAINED:
    print("Backbone freeze warmup: DISABLED (FREEZE_BACKBONE_EPOCHS = 0)")

# --- Main compile (always exactly once) ---
optimizer, scheduler = _create_main_optimizer_and_scheduler()

model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **AUGMENTATION_CONFIG.to_dict(),
)

print(f"\nOptimizer: {optimizer.__class__.__name__}")
for i, group in enumerate(optimizer.param_groups):
    num_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i+1}: lr={group['lr']:.2e}, wd={group['weight_decay']:.2e}, params={num_params:,}")
print("\nModel compiled!")
print("=" * 60)



MODEL COMPILATION

BACKBONE FREEZE WARMUP (3 epochs)
----------------------------------------
LINet compiled with AdamW optimizer, cross_entropy loss
  Learning rate: 1.00e-03
  Scheduler: None
  Device: cuda, AMP: True


Epoch 3/3: 100%|██████████| 76/76 [00:06<00:00, 12.40it/s, train_loss=2.5743, train_acc=0.2198, train_mca=0.2182, lr=1.00e-03]

Restored best weights from epoch 3 (train_loss=2.5743)

  All parameters unfrozen.
LINet compiled with AdamW optimizer, cross_entropy loss
  Using 6 parameter groups:
    Group 1: lr=1.27e-04, weight_decay=4.05e-04
    Group 2: lr=1.27e-04, weight_decay=4.05e-04
    Group 3: lr=4.10e-05, weight_decay=1.26e-03
    Group 4: lr=4.10e-05, weight_decay=1.26e-03
    Group 5: lr=4.10e-05, weight_decay=1.26e-03
    Group 6: lr=4.10e-05, weight_decay=0.00e+00
  Scheduler: SequentialLR
  Device: cuda, AMP: True

Optimizer: AdamW
  Group 1: lr=1.27e-04, wd=4.05e-04, params=7,056
  Group 2: lr=1.27e-04, wd=4.05e-04, params=2,352
  Group 3: lr=4.10e-05, wd=1.26e-03, params=6,283,296
  Group 4: lr=4.10e-05, wd=1.26e-03, params=6,283,296
  Group 5: lr=4.10e-05, wd=1.26e-03, params=2,749,075
  Group 6: lr=4.10e-05, wd=0.00e+00, params=7,200

Model compiled!


## 10b. Optional Backbone Freeze Warmup

If `LOAD_PRETRAINED = True` and `FREEZE_BACKBONE_EPOCHS > 0`, freeze the pretrained backbone and train only the classifier head for a few epochs. This lets the new head calibrate before the backbone starts adapting.

## 11. Train with Full Diagnostics

All diagnostics enabled: gradient health monitoring, per-stream training loss decomposition, integration weight norm tracking + periodic full snapshots, stream-specific accuracy monitoring.

In [15]:
def merge_histories(h1, h2):
    merged = {}
    all_keys = set(h1.keys()) | set(h2.keys())
    for key in all_keys:
        v1 = h1.get(key)
        v2 = h2.get(key)
        if v1 is None and v2 is None:
            merged[key] = None
        elif v1 is None:
            merged[key] = v2
        elif v2 is None:
            merged[key] = v1
        elif isinstance(v1, dict) and isinstance(v2, dict):
            # Recursively merge nested dicts (e.g. per-stream tracking)
            merged[key] = merge_histories(v1, v2)
        elif isinstance(v1, list) and isinstance(v2, list):
            merged[key] = v1 + v2
        else:
            # Scalar or unknown — just keep v2 (phase 2 wins)
            merged[key] = v2
    return merged

In [17]:
import warnings
import os

# Suppress PyTorch SequentialLR deprecation warning
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

# Create integration snapshot directory
os.makedirs(TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

print("=" * 60)
print("TRAINING WITH FULL DIAGNOSTICS")
print("=" * 60)

print(f"Configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")


print("=" * 60 + "\n")

# Train with all diagnostics enabled
history = model.fit(
    train_loader=train_loader,
    val_loader=test_loader,  # No validation set
    epochs=TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=TRAIN_CONFIG['save_path'],
    early_stopping=TRAIN_CONFIG['early_stopping'],
    restore_best_weights=True,  #TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=TRAIN_CONFIG['stream_monitoring'],
    monitor=TRAIN_CONFIG['monitor'],
    modality_dropout=TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=TRAIN_CONFIG['modality_dropout_rate'],
    # # Gradient health monitoring
    # gradient_monitoring=TRAIN_CONFIG['gradient_monitoring'],
    # gradient_log_freq=TRAIN_CONFIG['gradient_log_freq'],
    # # Integration weight tracking
    # track_integration_weights=TRAIN_CONFIG['track_integration_weights'],
    # integration_snapshot_path=TRAIN_CONFIG['integration_snapshot_path'],
    # integration_snapshot_freq=TRAIN_CONFIG['integration_snapshot_freq'],
)

# Merge warmup + full history if freeze warmup was used
if warmup_history is not None:
    history = merge_histories(warmup_history, history)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)

TRAINING WITH FULL DIAGNOSTICS
Configuration:
  epochs: 30
  grad_clip_norm: 0.9
  early_stopping: False
  restore_best_weights: False
  stream_monitoring: False
  modality_dropout: True
  modality_dropout_start: 0
  modality_dropout_ramp: 0
  modality_dropout_rate: 0.49
  label_smoothing: 0.06
  gradient_monitoring: False
  gradient_log_freq: 0
  track_integration_weights: False
  integration_snapshot_freq: 10
  monitor: val_mca
  save_path: /content/drive/MyDrive/linet_checkpoints/run_20260423_191732/best_model.pt
  integration_snapshot_path: /content/drive/MyDrive/linet_checkpoints/run_20260423_191732/integration_snapshots


🎲 Modality dropout activated at epoch 1 (prob=49.0%)
   Schedule: constant at 49%


Epoch 1/30: 100%|██████████| 149/149 [00:12<00:00, 11.49it/s, train_loss=2.4693, train_acc=0.2568, train_mca=0.2543, val_loss=2.1156, val_acc=0.4022, val_mca=0.3351, lr=4.10e-05]


  🎲 Dropout: 2356/4845 blanked (48.6%) | s0:23.9%, s1:24.7%


Epoch 2/30: 100%|██████████| 149/149 [00:07<00:00, 19.26it/s, train_loss=2.2640, train_acc=0.3032, train_mca=0.3051, val_loss=2.0065, val_acc=0.4449, val_mca=0.3938, lr=7.37e-05]


  🎲 Dropout: 2351/4845 blanked (48.5%) | s0:23.7%, s1:24.8%


...

  🎲 Dropout: 2371/4845 blanked (48.9%) | s0:24.7%, s1:24.3%


Epoch 29/30: 100%|██████████| 149/149 [00:07<00:00, 19.81it/s, train_loss=0.9239, train_acc=0.8402, train_mca=0.8378, val_loss=1.8357, val_acc=0.5246, val_mca=0.4777, lr=5.59e-06]


  🎲 Dropout: 2384/4845 blanked (49.2%) | s0:24.9%, s1:24.3%


Epoch 30/30: 100%|██████████| 149/149 [00:07<00:00, 20.39it/s, train_loss=0.9245, train_acc=0.8454, train_mca=0.8455, val_loss=1.8354, val_acc=0.5239, val_mca=0.4798, lr=3.21e-06]

  🎲 Dropout: 2376/4845 blanked (49.0%) | s0:25.3%, s1:23.8%

TRAINING COMPLETE!


## 12. Single-Stream Robustness Evaluation

How much does the model degrade when a stream is missing? Tests full model, RGB-only, and HHA-only.

In [18]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

# Evaluate with all streams (normal)
print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(test_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%  MCA: {results_both['mean_class_accuracy']*100:.2f}%")

# Evaluate with RGB only (HHA blanked)
print("\n[2/3] Evaluating with RGB ONLY (HHA blanked):")
results_rgb_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%  MCA: {results_rgb_only['mean_class_accuracy']*100:.2f}%")

# Evaluate with HHA only (RGB blanked)
print("\n[3/3] Evaluating with HHA ONLY (RGB blanked):")
results_depth_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%  MCA: {results_depth_only['mean_class_accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  Acc={results_both['accuracy']*100:.2f}%  MCA={results_both['mean_class_accuracy']*100:.2f}%")
print(f"  RGB only:      Acc={results_rgb_only['accuracy']*100:.2f}%  MCA={results_rgb_only['mean_class_accuracy']*100:.2f}% (HHA missing)")
print(f"  HHA only:      Acc={results_depth_only['accuracy']*100:.2f}%  MCA={results_depth_only['mean_class_accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100

print(f"\n  Degradation when HHA missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")
print("\n" + "=" * 60)


SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)

Testing model performance with missing streams...

[1/3] Evaluating with BOTH streams (normal):
      Accuracy: 52.39%  MCA: 47.98%

[2/3] Evaluating with RGB ONLY (Depth blanked):
      Accuracy: 41.19%  MCA: 38.05%

[3/3] Evaluating with DEPTH ONLY (RGB blanked):
      Accuracy: 43.59%  MCA: 40.32%

ROBUSTNESS SUMMARY

  Both streams:  Acc=52.39%  MCA=47.98%
  RGB only:      Acc=41.19%  MCA=38.05% (Depth missing)
  Depth only:    Acc=43.59%  MCA=40.32% (RGB missing)

  Degradation when Depth missing: +11.20%
  Degradation when RGB missing:   +8.80%



## 13. Test Set Evaluation + Pathway Analysis

In [19]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Evaluate on test set
results = model.evaluate(data_loader=test_loader, stream_monitoring=True)

print(f"\nTest Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")
print(f"  Mean Class Accuracy: {results['mean_class_accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

# Pathway analysis
print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")
print(f"\nAnalyzing stream pathways and integrated pathway contributions...")

pathway_analysis = model.analyze_pathways(data_loader=test_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

# Accuracy
print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")
# acc_int = pathway_analysis['accuracy']['integrated_only']
# contrib_int = pathway_analysis['accuracy']['integrated_contribution']
# print(f"  Integrated only: {acc_int*100:.2f}%  (contribution ratio: {contrib_int:.3f})")

# Loss
print("\nLoss:")
print(f"  Full model:      {pathway_analysis['loss']['full_model']:.4f}")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    loss_i = pathway_analysis['loss'][f'stream{i}_only']
    loss_contrib = pathway_analysis['loss'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {loss_i:.4f}  (loss ratio: {loss_contrib:.3f})")
# loss_int = pathway_analysis['loss']['integrated_only']
# loss_int_contrib = pathway_analysis['loss']['integrated_contribution']
# print(f"  Integrated only: {loss_int:.4f}  (loss ratio: {loss_int_contrib:.3f})")

# Feature norms
print("\nFeature Norms (mean +/- std):")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

# Training summary
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"  Initial train loss: {history['train_loss'][0]:.4f}")
print(f"  Final train loss:   {history['train_loss'][-1]:.4f}")
print(f"  Initial train acc:  {history['train_accuracy'][0]*100:.2f}%")
print(f"  Final train acc:    {history['train_accuracy'][-1]*100:.2f}%")
print(f"  Test accuracy:      {results['accuracy']*100:.2f}%")
print(f"  Test MCA:           {results['mean_class_accuracy']*100:.2f}%")
print(f"  Total epochs:       {len(history['train_loss'])}")

print("\n" + "=" * 60)

TEST SET EVALUATION

Test Results:
  Loss: 1.8354
  Overall Accuracy: 52.39%
  Mean Class Accuracy: 47.98%

Stream-Specific Performance:
  Stream0 (RGB) Solo Accuracy: 41.19%
  Stream0 (RGB) Contribution: +8.80%
  Stream1 (Depth) Solo Accuracy: 43.59%
  Stream1 (Depth) Contribution: +11.20%

PATHWAY ANALYSIS

Analyzing stream pathways and integrated pathway contributions...

Samples analyzed: 4659

Accuracy:
  Full model:      52.39%
  RGB only:       43.59%  (contribution ratio: 0.088)
  Depth only:       41.19%  (contribution ratio: 0.112)

Loss:
  Full model:      1.8354
  RGB only:       2.0470  (loss ratio: 1.115)
  Depth only:       2.1236  (loss ratio: 1.157)

Feature Norms (mean +/- std):
  RGB:        82.5122 +/- 21.9391
  Depth:        66.4841 +/- 18.2555
  Integrated:  13.6012 +/- 2.3681

TRAINING SUMMARY
  Initial train loss: 2.8257
  Final train loss:   0.9245
  Initial train acc:  12.09%
  Final train acc:    84.54%
  Test accuracy:      52.39%
  Test MCA:           47.98

## 13b. Integration Saturation Analysis (Stable Rank vs Baselines)

Tests whether the trained `integration_from_streams` weights have spread their
energy across the full integrated-stream capacity, or whether gradient descent
collapsed them into a low-dim subspace — the question being: *is the fusion
actually a bottleneck under HHA, or is the bottleneck elsewhere?*

**Headline metric: stable rank**
$$
\text{stable\_rank}(W) = \frac{\|W\|_F^2}{\|W\|_2^2} = \frac{\sum_i s_i^2}{s_{\max}^2}
$$

Continuous, energy-weighted "effective dimensionality." Range: 1 (rank-1 matrix,
all energy in one direction) to `min(rows, cols)` (energy distributed evenly).
Avoids the "long tail of barely-nonzero singular values that aren't really
doing anything" problem of threshold-counted rank.

**Saturation = stable_rank / min(rows, cols)** ∈ (0, 1].

**Baselines** — without these, a single saturation number is uninterpretable.

1. **Random Gaussian baseline** (always computed). For each weight shape, sample
   N i.i.d. Gaussian matrices and compute the mean stable rank. For a square
   N×N i.i.d. Gaussian, stable rank ≈ N/4 (saturation ≈ 0.25). The trained
   weight's saturation **above** this = optimizer pushed to spread energy more
   than chance ⇒ saturation pressure. **Below** this = optimizer concentrated
   energy in a low-dim subspace ⇒ slack.
2. **Cross-model baseline** (optional). Set `BASELINE_CHECKPOINT_PATH` below to
   the path of a raw-depth-trained model with the same architecture; the cell
   will overlay its saturation curve. Difference between HHA and raw-depth
   saturation at the same layer is the most direct evidence of HHA-specific
   fusion pressure (or its absence).

**Decision rule** (the only one that's truly grounded):
- HHA late-layer stable rank > raw-depth late-layer stable rank by a meaningful
  margin → HHA is denser through the fusion than raw-depth was; widening
  `integrated_inplanes` will likely help. **Run the widen ablation.**
- HHA ≈ raw-depth → not HHA-specific pressure. Widening is independent of the
  HHA decision; some other bottleneck is dominant.
- HHA < raw-depth → HHA is more compressible than raw-depth (unexpected but
  possible if HHA is dominated by a few semantic axes). Widening is wasted compute.

If you don't have a raw-depth checkpoint, the random-Gaussian baseline gives a
weaker but still meaningful signal: *did the optimizer demand more spread than
random?*


In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import torch
from collections import defaultdict


# ============================================================================
# Optional cross-model baseline. If set, the cell loads it and overlays its
# saturation curve. Use a raw-depth-trained checkpoint of the same architecture
# (e.g. SUN raw-depth fine-tuning, or this same notebook trained with
# stream_input_channels=[3, 1] and use_hha=False).
# Set to None to skip the cross-model comparison.
# ============================================================================
BASELINE_CHECKPOINT_PATH = '/content/drive/MyDrive/linet_checkpoints/run_20260410_145529_MD_45.2%/final_model.pt'
BASELINE_LABEL = "raw-depth"

# Number of Gaussian samples per shape for the random baseline.
# Sampling cap; bumped automatically for larger matrices below since 50 samples
# get noisier as min(rows, cols) grows.
GAUSSIAN_BASELINE_SAMPLES_BASE = 50
GAUSSIAN_RNG = np.random.default_rng(seed=0)
# Cache shared across both _analyze() calls so the same shape gives identical
# baseline_stable values for the HHA model and the cross-model baseline.
_GAUSSIAN_CACHE: dict[tuple, float] = {}


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def _stable_rank_from_S(S):
    """Stable rank: sum(s_i^2) / s_max^2. Continuous, energy-weighted."""
    if S.size == 0:
        return 0.0
    s_max = float(S[0])
    if s_max <= 0:
        return 0.0
    return float((S ** 2).sum() / (s_max ** 2))


def _gaussian_baseline_stable_rank(shape):
    """Mean stable rank of i.i.d. N(0,1) matrices of the given shape.
    Sample count scales with matrix size to keep the empirical estimate stable
    — stable rank for a 512x512 i.i.d. Gaussian has more variance per draw than
    for a 64x64. Cached implicitly via the calling site below."""
    out, in_ = shape
    # Heuristic: at least 50 samples; double again every time max_dim doubles past 64.
    max_dim = max(out, in_)
    n_samples = GAUSSIAN_BASELINE_SAMPLES_BASE * max(1, int(np.ceil(max_dim / 64)))
    vals = []
    for _ in range(n_samples):
        M = GAUSSIAN_RNG.standard_normal(size=(out, in_))
        s = np.linalg.svd(M, compute_uv=False)
        vals.append(_stable_rank_from_S(s))
    return float(np.mean(vals))


def _collect_integration_weights(m):
    """Walk the model; yield (layer_path, stream_idx, W_2d_numpy)."""
    for name, module in m.named_modules():
        if not hasattr(module, 'integration_from_streams'):
            continue
        if not isinstance(module.integration_from_streams, torch.nn.ParameterList):
            continue
        for stream_idx, p in enumerate(module.integration_from_streams):
            w = p.detach().cpu().squeeze(-1).squeeze(-1)
            if w.numel() == 0:
                continue
            yield name, stream_idx, w.numpy()


def _layer_depth_order(layer_path):
    if layer_path == 'conv1':
        return (0, 0, '')
    parts = layer_path.split('.')
    stage = parts[0]
    depth_idx = {'conv1': 0, 'layer1': 1, 'layer2': 2, 'layer3': 3, 'layer4': 4}.get(stage, 99)
    return (depth_idx, len(parts), layer_path)


def _analyze(model_to_analyze):
    """Return list of dicts (one per layer×stream) with all metrics."""
    rows = []
    spectra = []
    for layer_path, stream_idx, W in _collect_integration_weights(model_to_analyze):
        S = np.linalg.svd(W, compute_uv=False)
        s_max = float(S[0])
        if s_max <= 0:
            continue
        stable_rank = _stable_rank_from_S(S)
        max_possible = float(min(W.shape))
        # Threshold-counted ranks (supporting numbers, not headline).
        rank_1pct = int((S > 0.01 * s_max).sum())
        rank_5pct = int((S > 0.05 * s_max).sum())
        # Random Gaussian baseline (cached by shape).
        shape_key = tuple(W.shape)
        if shape_key not in _GAUSSIAN_CACHE:
            _GAUSSIAN_CACHE[shape_key] = _gaussian_baseline_stable_rank(shape_key)
        baseline_stable = _GAUSSIAN_CACHE[shape_key]
        # Condition number with numerical-zero handling.
        s_min = float(S[-1])
        if s_min < 1e-9:
            cond_str = "near-singular"
        else:
            cond_str = f"{s_max / s_min:>9.1f}"
        rows.append({
            'layer': layer_path,
            'stream_idx': stream_idx,
            'stream_label': STREAM_LABELS.get(stream_idx, f'stream_{stream_idx}'),
            'shape': shape_key,
            'max_possible': max_possible,
            's_max': s_max,
            's_min': s_min,
            'stable_rank': stable_rank,
            'sat_stable': stable_rank / max_possible,
            'rank_1pct': rank_1pct,
            'rank_5pct': rank_5pct,
            'baseline_stable': baseline_stable,
            'baseline_sat': baseline_stable / max_possible,
            'gap_vs_baseline': stable_rank - baseline_stable,
            'cond_str': cond_str,
        })
        spectra.append((f"{layer_path}.stream{stream_idx}", S / s_max))
    rows.sort(key=lambda r: (_layer_depth_order(r['layer']), r['stream_idx']))
    return rows, spectra


# ----------------------------------------------------------------------------
# Analyze the loaded (HHA-trained) model.
# ----------------------------------------------------------------------------
print("Analyzing HHA-trained model integration weights...")
rows, spectra = _analyze(model)

# ----------------------------------------------------------------------------
# Optionally analyze the cross-model baseline (raw-depth, same architecture).
# ----------------------------------------------------------------------------
baseline_rows = None
if BASELINE_CHECKPOINT_PATH is not None:
    print(f"\nLoading cross-model baseline checkpoint:\n  {BASELINE_CHECKPOINT_PATH}")
    # Construct a model of the same architecture and load the checkpoint.
    # Note: the baseline checkpoint may have stream_input_channels=[3, 1] (raw depth).
    # That only changes the conv1 stream_weights[1] shape; integration weights
    # are identical in shape between HHA and raw-depth models, so SVD is meaningful.
    baseline_model = copy.deepcopy(model)
    # Try to load the checkpoint; skip mismatched keys (the conv1 stream-1 stem
    # will be one of them if the baseline was trained on 1ch raw depth).
    try:
        # weights_only=False is deliberate: the checkpoint contains a config dict
        # alongside the state_dict (history, optimizer state, etc.), not just tensors.
        # Newer PyTorch defaults flip toward weights_only=True; suppress the warning
        # by being explicit.
        ckpt = torch.load(BASELINE_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        if 'model_state_dict' in ckpt:
            sd = ckpt['model_state_dict']
        elif 'state_dict' in ckpt:
            sd = ckpt['state_dict']
        else:
            sd = ckpt
        # Build the baseline model with [3, 1] stream channels so its state_dict matches.
        # Easiest: load with strict=False onto the deepcopy, accept skip on stream_weights.1.
        # But the deepcopy already has [3, 3] stem so loading a [3, 1] stem will skip.
        # That's fine — we don't need the stem for this analysis, only integration weights.
        own = baseline_model.state_dict()
        loadable = {k: v for k, v in sd.items() if k in own and v.shape == own[k].shape}
        baseline_model.load_state_dict(loadable, strict=False)
        skipped = [k for k in sd.keys() if k not in loadable]
        print(f"  Loaded {len(loadable)} / {len(sd)} keys "
              f"(skipped {len(skipped)} due to shape mismatch — expected for HHA->raw-depth stem swap)")
        # Verify integration weights specifically loaded — those are what this
        # analysis depends on. Stem mismatch is fine; integration mismatch invalidates the comparison.
        n_int_loaded = sum(1 for k in own if 'integration_from_streams' in k and k in loadable)
        n_int_total = sum(1 for k in own if 'integration_from_streams' in k)
        print(f"  Integration weights loaded: {n_int_loaded} / {n_int_total}")
        if n_int_loaded < n_int_total:
            print("  WARNING: not all integration weights loaded — baseline analysis is partial.")
            print("  (different width_multiplier / num_blocks / num_classes between models?)")
        baseline_rows, _ = _analyze(baseline_model)
    except Exception as exc:
        print(f"  WARNING: failed to load baseline checkpoint: {exc}")
        baseline_rows = None


# ----------------------------------------------------------------------------
# Print table — stable rank as headline.
# ----------------------------------------------------------------------------
print("\nIntegration weight stable-rank analysis (post-training)")
print("=" * 130)
print(f"{'layer':<32} {'stream':<6} {'shape':<12} "
      f"{'s_max':>7} "
      f"{'stable':>7} {'sat':>5} "
      f"{'baseline':>9} {'gap':>6} "
      f"{'r@1%':>5} {'r@5%':>5} "
      f"{'cond':>14}")
print("-" * 130)
for r in rows:
    print(f"{r['layer']:<32} {r['stream_label']:<6} {str(r['shape']):<12} "
          f"{r['s_max']:>7.3f} "
          f"{r['stable_rank']:>7.2f} {r['sat_stable']:>5.2f} "
          f"{r['baseline_stable']:>9.2f} {r['gap_vs_baseline']:>+6.2f} "
          f"{r['rank_1pct']:>5d} {r['rank_5pct']:>5d} "
          f"{r['cond_str']:>14}")

# ----------------------------------------------------------------------------
# Plots.
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- Plot 1: stable-rank saturation by layer depth, with Gaussian baseline ---
ax = axes[0]
streams = sorted({r['stream_idx'] for r in rows})
unique_layers = []
for r in rows:
    if r['layer'] not in unique_layers:
        unique_layers.append(r['layer'])
x_pos = {layer: i for i, layer in enumerate(unique_layers)}

# HHA model curves.
for s_idx in streams:
    sub = [r for r in rows if r['stream_idx'] == s_idx]
    label = f"HHA model {STREAM_LABELS.get(s_idx, str(s_idx))}"
    xs = [x_pos[r['layer']] for r in sub]
    ys = [r['sat_stable'] for r in sub]
    ax.plot(xs, ys, 'o-', label=label, linewidth=2, markersize=6, alpha=0.9)

# Random-Gaussian baseline (one curve, since baseline is shape-only).
baseline_curve = []
xs_bl = []
seen = set()
for r in rows:
    key = r['layer']  # one point per layer; baseline is the same per shape
    if key in seen: continue
    seen.add(key)
    xs_bl.append(x_pos[key])
    baseline_curve.append(r['baseline_sat'])
ax.plot(xs_bl, baseline_curve, 's--', color='gray', alpha=0.7, linewidth=1.5,
        label=f'Gaussian baseline (mean {np.mean(baseline_curve):.2f})')

# Optional cross-model baseline curves.
if baseline_rows is not None:
    for s_idx in streams:
        sub = [r for r in baseline_rows if r['stream_idx'] == s_idx]
        if not sub: continue
        label = f"{BASELINE_LABEL} {STREAM_LABELS.get(s_idx, str(s_idx))}"
        xs = [x_pos[r['layer']] for r in sub if r['layer'] in x_pos]
        ys = [r['sat_stable'] for r in sub if r['layer'] in x_pos]
        ax.plot(xs, ys, '^:', linewidth=1.5, markersize=5, alpha=0.7, label=label)

ax.set_xlabel('Layer (network depth →)', fontsize=11)
ax.set_ylabel('Saturation = stable_rank / min(in, out)', fontsize=11)
ax.set_title('Stable-rank saturation by layer\n'
             '[HHA above baseline = saturation pressure; below = slack]',
             fontsize=11)
ax.set_xticks(list(x_pos.values()))
ax.set_xticklabels(list(x_pos.keys()), rotation=70, fontsize=7)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# --- Plot 2: late-layer (layer4) singular-value spectra with stratified sampling ---
ax = axes[1]
late = [(lab, S) for lab, S in spectra if lab.startswith('layer4')]
# Stratified by block: keep ALL entries from each layer4.<block>.xxx subgroup.
# For ResNet-18, layer4 has 2 blocks * 2 conv layers/block * num_streams; not large.
# We avoid the [:10] silent-cap pitfall by including everything explicitly.
if len(late) > 16:
    # Forward-defensive for deeper LINet variants. ResNet-18's layer4 has at most
    # 2 blocks * 2 conv layers/block * num_streams = 8 entries, so this branch is
    # dead code for the current architecture; included so plot doesn't silently
    # truncate if a deeper variant is loaded.
    by_block = defaultdict(list)
    for lab, S in late:
        # Extract the block index from "layer4.<block>.<rest>"
        block_id = lab.split('.')[1] if '.' in lab else 'all'
        by_block[block_id].append((lab, S))
    late = []
    for block_id, lst in by_block.items():
        late.extend(lst[:4])

cmap = plt.cm.viridis
for i, (label, S_norm) in enumerate(late):
    ax.semilogy(S_norm, alpha=0.85, linewidth=1.5,
                color=cmap(i / max(1, len(late) - 1)),
                label=label.replace('layer4.', ''))
ax.axhline(y=0.01, color='red', linestyle=':', alpha=0.5, label='1% threshold')
ax.axhline(y=0.05, color='orange', linestyle=':', alpha=0.5, label='5% threshold')
ax.set_xlabel('Singular-value index', fontsize=11)
ax.set_ylabel('Normalized singular value (s_i / s_max)', fontsize=11)
ax.set_title('Late-layer (layer4) singular-value spectra\n'
             '[Slow decay = energy spread; sharp drop = concentrated]',
             fontsize=11)
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(1e-4, 2.0)

# --- Plot 3: gap-vs-Gaussian-baseline as FRACTION OF CAPACITY ---
# Absolute stable-rank units mix layers of different widths (64 at conv1,
# 512 at layer4). +5 at width 64 means 8% of capacity; +5 at width 512
# means 1%. Normalize by max_possible so layers are comparable.
ax = axes[2]
for s_idx in streams:
    sub = [r for r in rows if r['stream_idx'] == s_idx]
    label = f"HHA {STREAM_LABELS.get(s_idx, str(s_idx))}"
    xs = [x_pos[r['layer']] for r in sub]
    ys = [r['gap_vs_baseline'] / r['max_possible'] for r in sub]
    ax.plot(xs, ys, 'o-', linewidth=2, markersize=6, alpha=0.9, label=label)
if baseline_rows is not None:
    for s_idx in streams:
        sub = [r for r in baseline_rows if r['stream_idx'] == s_idx and r['layer'] in x_pos]
        if not sub: continue
        label = f"{BASELINE_LABEL} {STREAM_LABELS.get(s_idx, str(s_idx))}"
        xs = [x_pos[r['layer']] for r in sub]
        ys = [r['gap_vs_baseline'] / r['max_possible'] for r in sub]
        ax.plot(xs, ys, '^:', linewidth=1.5, markersize=5, alpha=0.7, label=label)
ax.axhline(y=0.0, color='gray', linewidth=1.0, alpha=0.5)
ax.set_xlabel('Layer (network depth →)', fontsize=11)
ax.set_ylabel('(stable_rank − baseline) / max_possible', fontsize=11)
ax.set_title('Gap vs Gaussian baseline, as fraction of capacity\n'
             '[Above 0 = optimizer demanded more spread than random; comparable across layers]',
             fontsize=11)
ax.set_xticks(list(x_pos.values()))
ax.set_xticklabels(list(x_pos.keys()), rotation=70, fontsize=7)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = f"{checkpoint_dir}/integration_saturation_svd.png"
plt.savefig(out_path, dpi=110, bbox_inches='tight')
plt.show()
print(f"\nSaved figure: {out_path}")


# ----------------------------------------------------------------------------
# Verdict — anchored to baselines, not absolute thresholds.
# ----------------------------------------------------------------------------
late_rows = [r for r in rows if r['layer'].startswith('layer4')]
if late_rows:
    avg_late_stable = float(np.mean([r['stable_rank'] for r in late_rows]))
    avg_late_baseline = float(np.mean([r['baseline_stable'] for r in late_rows]))
    avg_late_gap = avg_late_stable - avg_late_baseline
    avg_late_max = float(np.mean([r['max_possible'] for r in late_rows]))
    print(f"\n=== Late-layer (layer4) summary ===")
    print(f"  HHA model   stable rank: {avg_late_stable:.2f} / max {avg_late_max:.0f}")
    print(f"  Gauss base. stable rank: {avg_late_baseline:.2f}")
    print(f"  Gap (HHA − Gaussian):   {avg_late_gap:+.2f}")

    # Only the cross-model comparison gives a strong verdict; against Gaussian
    # alone, the verdict is qualified.
    if baseline_rows is not None:
        bl_late = [r for r in baseline_rows if r['layer'].startswith('layer4')]
        if bl_late:
            avg_bl_stable = float(np.mean([r['stable_rank'] for r in bl_late]))
            margin = avg_late_stable - avg_bl_stable
            print(f"  {BASELINE_LABEL} model  stable rank: {avg_bl_stable:.2f}")
            print(f"  Margin (HHA − {BASELINE_LABEL}):  {margin:+.2f}")
            print(f"\n=== Verdict (cross-model baseline) ===")
            margin_pct = (margin / max(avg_bl_stable, 1e-6)) * 100
            if margin > 0 and margin_pct > 10:
                print(f"  >> HHA pushes the fusion {margin_pct:+.0f}% harder than {BASELINE_LABEL}")
                print(f"     at late layers. Widening integrated_inplanes (1.5x) is")
                print(f"     justified — run the ablation.")
            elif margin > 0 and margin_pct > 3:
                print(f"  >> HHA marginally above {BASELINE_LABEL} ({margin_pct:+.0f}%).")
                print(f"     Widening might give a small bump; ablation optional.")
            elif abs(margin_pct) <= 3:
                print(f"  >> HHA ≈ {BASELINE_LABEL} at the fusion stage (within ±3%).")
                print(f"     Not HHA-specific pressure; widening unlikely to help.")
            else:
                print(f"  >> HHA below {BASELINE_LABEL} ({margin_pct:+.0f}%).")
                print(f"     HHA produces more compressible features than raw-depth.")
                print(f"     Widening is wasted compute.")
    else:
        # No cross-model baseline -> 'is this saturated?' is genuinely undecidable.
        # The Gaussian baseline is a passive-entropy floor, not a capacity-bottleneck
        # reference. Print the numbers for transparency but DON'T pretend they answer
        # the decision question — that requires a trained reference model.
        avg_late_gap_frac = avg_late_gap / avg_late_max
        print(f"\n=== Verdict (no cross-model baseline — UNDECIDABLE) ===")
        print(f"  Late-layer HHA stable rank differs from the random-Gaussian baseline")
        print(f"  by {avg_late_gap:+.2f} absolute units ({avg_late_gap_frac*100:+.1f}% of capacity).")
        print(f"  Optimizer demanded {'more' if avg_late_gap > 0 else 'less'} spread than chance.")
        print(f"")
        print(f"  But 'is this saturated enough to widen?' is genuinely undecidable")
        print(f"  without a reference trained model. The Gaussian baseline is a passive-")
        print(f"  entropy floor (any trained matrix tends to exceed it slightly); it is")
        print(f"  not a capacity-bottleneck reference.")
        print(f"")
        print(f"  RECOMMENDED ACTION: set BASELINE_CHECKPOINT_PATH at the top of this")
        print(f"  cell to a raw-depth-trained model with the same architecture, and")
        print(f"  re-run. The HHA-vs-baseline margin is the only actionable signal.")

    # Per-stream comparison — only meaningful relative to the baseline gap, not
    # an absolute threshold.
    if len(streams) == 2:
        sat_per_stream = {}
        for s_idx in streams:
            srows = [r for r in late_rows if r['stream_idx'] == s_idx]
            if srows:
                sat_per_stream[STREAM_LABELS.get(s_idx, str(s_idx))] = float(
                    np.mean([r['gap_vs_baseline'] for r in srows])
                )
        if len(sat_per_stream) == 2:
            labels = list(sat_per_stream.keys())
            gaps = list(sat_per_stream.values())
            asym = gaps[1] - gaps[0]
            # Print as fraction-of-capacity to match plot 3's units (and the
            # delta_asym_frac threshold below). Absolute stable-rank units are
            # not comparable across layers of different widths.
            gaps_frac = [g / avg_late_max for g in gaps]
            print(f"\n  Per-stream gap-vs-Gaussian (% of layer capacity): "
                  f"{labels[0]}={gaps_frac[0]*100:+.1f}%, {labels[1]}={gaps_frac[1]*100:+.1f}%")
            if baseline_rows is not None:
                bl_sat = {}
                bl_late_rows = [r for r in baseline_rows if r['layer'].startswith('layer4')]
                for s_idx in streams:
                    srows = [r for r in bl_late_rows if r['stream_idx'] == s_idx]
                    if srows:
                        bl_sat[STREAM_LABELS.get(s_idx, str(s_idx))] = float(
                            np.mean([r['gap_vs_baseline'] for r in srows])
                        )
                if len(bl_sat) == 2:
                    bl_asym = list(bl_sat.values())[1] - list(bl_sat.values())[0]
                    bl_gaps_frac = [g / avg_late_max for g in bl_sat.values()]
                    print(f"  {BASELINE_LABEL} model per-stream gap-vs-Gaussian "
                          f"(% of layer capacity): "
                          f"{labels[0]}={bl_gaps_frac[0]*100:+.1f}%, "
                          f"{labels[1]}={bl_gaps_frac[1]*100:+.1f}%")
                    delta_asym = asym - bl_asym
                    # Threshold on delta_asym as a fraction of layer capacity, not
                    # absolute stable-rank units — a 1.0 difference is huge at
                    # width 64 but trivial at width 512.
                    delta_asym_frac = abs(delta_asym) / avg_late_max
                    if delta_asym_frac > 0.05:
                        more = labels[1] if delta_asym > 0 else labels[0]
                        print(f"  {more} stream's HHA-vs-{BASELINE_LABEL} asymmetry exceeds")
                        print(f"  the {BASELINE_LABEL} baseline's natural asymmetry by")
                        print(f"  {delta_asym_frac*100:+.1f}% of layer capacity — suggests {more}")
                        print(f"  carries HHA-specific information density that {BASELINE_LABEL} doesn't.")


## 13c. Integration Weight Rank-1 Collapse — Diagnosis

The 13b table showed `stable_rank ≈ 1.0` for **every** integration weight in **both**
the HHA and raw-depth models. That is suspicious: a healthy 1×1 conv almost never
collapses to rank-1 across every layer of two independently-trained networks.

Smoking-gun check: at init, `integration_from_streams[i]` is `c · ones(M, N)` with
`c = 1/num_streams = 0.5` (see `src/models/linear_integration/li_net3/conv.py:281`).
A rank-1 constant matrix `c·ones(M, N)` has a single nonzero singular value
`s_1 = c · sqrt(M · N)`. For square shapes that's `c · M`:

  conv1   (48,  48):  predicted 0.5 × 48  = 24    observed: 23.85 - 23.97  ✓
  layer1  (48,  48):  predicted 0.5 × 48  = 24    observed: 23.85 - 23.97  ✓
  layer4 (384, 384):  predicted 0.5 × 384 = 192   observed: 191.18 - 191.81 ✓

Within ~0.5%. The integration weights are essentially still at constant init.

This cell pins down WHY by measuring:
  [A] distance from `W_init = c · ones`, ones-direction cosine, top SV ratios
  [B] alignment of the top singular **vectors** with the all-ones direction
  [C] stable rank of the residual `W - W_init` — what training actually added
  [D] cross-component stable-rank of `stream_weights` and `integrated_weight`
       (those use Kaiming/orthogonal init — should be healthy if training worked)
  [E] bias magnitudes — if cross-stream coupling moved into `integrated_bias`,
       biases will be large relative to the residual weight contributions


In [ ]:
import numpy as np
import torch


# ----------------------------------------------------------------------------
# Setup
# ----------------------------------------------------------------------------
# Defensive: if the cell is run in a notebook scope where STREAM_LABELS hasn't
# been defined, fall back to numeric labels so the table still prints.
if 'STREAM_LABELS' not in dir():
    STREAM_LABELS = {0: 'stream0', 1: 'stream1'}
def _get_num_streams(m):
    for mod in m.modules():
        ifs = getattr(mod, 'integration_from_streams', None)
        if isinstance(ifs, torch.nn.ParameterList):
            return len(ifs)
    raise RuntimeError("No integration_from_streams found in model")

NUM_STREAMS = _get_num_streams(model)
C_INIT = 1.0 / NUM_STREAMS
print(f"num_streams = {NUM_STREAMS}, constant init c = 1/{NUM_STREAMS} = {C_INIT}\n")


def _iter_int_weights(m):
    for name, module in m.named_modules():
        ifs = getattr(module, 'integration_from_streams', None)
        if not isinstance(ifs, torch.nn.ParameterList):
            continue
        for stream_idx, p in enumerate(ifs):
            w = p.detach().cpu().squeeze(-1).squeeze(-1)
            if w.numel() == 0:
                continue
            yield name, stream_idx, w


def _stable_rank_2d(W2d):
    if W2d.size == 0:
        return float('nan'), 0
    S = np.linalg.svd(W2d, compute_uv=False)
    s_max = float(S[0]) if len(S) > 0 else 0.0
    if s_max <= 0:
        return float('nan'), int(min(W2d.shape))
    return float((S ** 2).sum() / (s_max ** 2)), int(min(W2d.shape))


# ----------------------------------------------------------------------------
# [A] Per-layer: distance from constant init, ones-direction cosine, s_2/s_1
# ----------------------------------------------------------------------------
print("=" * 142)
print("[A] integration_from_streams: distance from W_init = c*ones, ones-direction alignment, top-SV ratios")
print("=" * 142)
print(f"{'layer':<32} {'stream':<6} {'shape':<14} "
      f"{'mean':>8} {'std':>8} {'||W||':>8} "
      f"{'rel_dist':>10} {'cos(W,ones)':>13} "
      f"{'s_1':>8} {'pred s_1':>10} {'s_2/s_1':>9}")
print("-" * 142)

A_rows = []
for name, stream_idx, w in _iter_int_weights(model):
    M, N = w.shape
    W_init = C_INIT * torch.ones(M, N)
    delta = w - W_init
    rel_dist = (delta.norm() / W_init.norm()).item()

    w_flat = w.flatten()
    cos_ones = float(
        (w_flat / w_flat.norm()).dot(torch.ones_like(w_flat) / np.sqrt(w_flat.numel()))
    )

    S = np.linalg.svd(w.numpy(), compute_uv=False)
    s1 = float(S[0])
    s1_pred = C_INIT * np.sqrt(M * N)
    s2_ratio = float(S[1] / s1) if (len(S) > 1 and s1 > 0) else float('nan')

    label = STREAM_LABELS.get(stream_idx, f"s{stream_idx}")
    print(f"{name:<32} {label:<6} {str(tuple(w.shape)):<14} "
          f"{w.mean().item():>8.4f} {w.std().item():>8.4f} {w.norm().item():>8.3f} "
          f"{rel_dist:>10.4f} {cos_ones:>13.4f} "
          f"{s1:>8.3f} {s1_pred:>10.3f} {s2_ratio:>9.4f}")
    A_rows.append({'layer': name, 'stream_idx': stream_idx, 'w': w,
                   'rel_dist': rel_dist, 'cos_ones': cos_ones, 's2_ratio': s2_ratio})


# ----------------------------------------------------------------------------
# [B] Top singular vectors vs all-ones direction
# ----------------------------------------------------------------------------
print()
print("=" * 142)
print("[B] Top singular VECTORS vs all-ones direction (|cos| close to 1 = dominant component is the constant-init mode)")
print("=" * 142)
print(f"{'layer':<32} {'stream':<6} {'|cos(u_1, ones)|':>20} {'|cos(v_1, ones)|':>20} {'product':>10}")
print("-" * 142)
for d in A_rows:
    w_np = d['w'].numpy()
    M, N = w_np.shape
    U, _S, Vt = np.linalg.svd(w_np, full_matrices=False)
    u1, v1 = U[:, 0], Vt[0, :]
    cos_u = float(abs(np.dot(u1, np.ones(M) / np.sqrt(M))))
    cos_v = float(abs(np.dot(v1, np.ones(N) / np.sqrt(N))))
    # Product = fraction of s_1 energy explained by the constant-init mode
    # (W = c*u_ones * v_ones^T projects entirely along u_1=v_1=ones; product=1).
    product = cos_u * cos_v
    label = STREAM_LABELS.get(d['stream_idx'], f"s{d['stream_idx']}")
    print(f"{d['layer']:<32} {label:<6} {cos_u:>20.4f} {cos_v:>20.4f} {product:>10.4f}")


# ----------------------------------------------------------------------------
# [C] Stable rank of residual W - W_init
# ----------------------------------------------------------------------------
# Decomposes the trained weight as W = W_init + residual. The residual is what
# training actually added on top of the symmetric init. Two regimes:
#   stable_rank(resid) ~ 1  -> training also moved along a single rank-1 mode
#   stable_rank(resid) > 1  -> training added genuine multi-dim structure but
#                              its norm is small relative to W_init, so the
#                              full SVD of W is dominated by the constant mode.
# ----------------------------------------------------------------------------
print()
print("=" * 142)
print("[C] Stable rank of residual (W - W_init) — what training added on top of symmetric init")
print("=" * 142)
print(f"{'layer':<32} {'stream':<6} {'||resid||':>12} {'stable_rank(resid)':>20} "
      f"{'sat(resid)':>12} {'||resid||/||W||':>17}")
print("-" * 142)
C_rows = []
for d in A_rows:
    w_np = d['w'].numpy()
    M, N = w_np.shape
    W_init_np = C_INIT * np.ones((M, N))
    delta = w_np - W_init_np
    delta_norm = float(np.linalg.norm(delta))
    w_norm = float(np.linalg.norm(w_np))
    if delta_norm < 1e-10:
        sr_str = '~0'
        sat_str = '—'
        sr_val = float('nan')
    else:
        S_d = np.linalg.svd(delta, compute_uv=False)
        s_max_d = float(S_d[0])
        if s_max_d < 1e-12:
            sr_str = '—'; sat_str = '—'; sr_val = float('nan')
        else:
            sr_val = float((S_d ** 2).sum() / (s_max_d ** 2))
            sr_str = f"{sr_val:.2f}"
            sat_str = f"{sr_val / min(M, N):.4f}"
    rel_to_w = delta_norm / w_norm if w_norm > 0 else float('nan')
    label = STREAM_LABELS.get(d['stream_idx'], f"s{d['stream_idx']}")
    print(f"{d['layer']:<32} {label:<6} {delta_norm:>12.4f} {sr_str:>20} "
          f"{sat_str:>12} {rel_to_w:>17.4f}")
    C_rows.append({'layer': d['layer'], 'stream_idx': d['stream_idx'],
                   'resid_sr': sr_val, 'rel_to_w': rel_to_w})


# ----------------------------------------------------------------------------
# [D] Cross-component stable rank — did the OTHER LIConv2d weights train?
# ----------------------------------------------------------------------------
print()
print("=" * 142)
print("[D] Cross-component stable rank: stream_weights (kaiming) and integrated_weight (orthogonal/kaiming)")
print("    Healthy training => these should show much higher stable rank than integration_from_streams.")
print("=" * 142)
print(f"{'layer':<32} {'component':<26} {'stream':<6} {'shape':<24} "
      f"{'stable_rank':>12} {'sat':>6}")
print("-" * 142)

D_rows = []
for name, module in model.named_modules():
    ifs = getattr(module, 'integration_from_streams', None)
    if not isinstance(ifs, torch.nn.ParameterList):
        continue

    sw = getattr(module, 'stream_weights', None)
    if isinstance(sw, torch.nn.ParameterList):
        for s_idx, p in enumerate(sw):
            w = p.detach().cpu()
            if w.numel() == 0:
                continue
            w2d = w.reshape(w.shape[0], -1).numpy()
            sr, mp = _stable_rank_2d(w2d)
            sat = sr / mp if mp > 0 else float('nan')
            label = STREAM_LABELS.get(s_idx, f"s{s_idx}")
            print(f"{name:<32} {'stream_weights':<26} {label:<6} {str(tuple(w.shape)):<24} "
                  f"{sr:>12.2f} {sat:>6.2f}")
            D_rows.append({'layer': name, 'component': 'stream_weights',
                           'sat': sat, 'sr': sr})

    for s_idx, p in enumerate(ifs):
        w = p.detach().cpu().squeeze(-1).squeeze(-1)
        if w.numel() == 0:
            continue
        sr, mp = _stable_rank_2d(w.numpy())
        sat = sr / mp if mp > 0 else float('nan')
        label = STREAM_LABELS.get(s_idx, f"s{s_idx}")
        print(f"{name:<32} {'integration_from_streams':<26} {label:<6} {str(tuple(w.shape)):<24} "
              f"{sr:>12.2f} {sat:>6.2f}")
        D_rows.append({'layer': name, 'component': 'integration_from_streams',
                       'sat': sat, 'sr': sr})

    iw = getattr(module, 'integrated_weight', None)
    if isinstance(iw, torch.nn.Parameter) and iw.numel() > 0:
        w = iw.detach().cpu()
        w2d = w.reshape(w.shape[0], -1).numpy()
        sr, mp = _stable_rank_2d(w2d)
        sat = sr / mp if mp > 0 else float('nan')
        print(f"{name:<32} {'integrated_weight':<26} {'-':<6} {str(tuple(w.shape)):<24} "
              f"{sr:>12.2f} {sat:>6.2f}")
        D_rows.append({'layer': name, 'component': 'integrated_weight',
                       'sat': sat, 'sr': sr})


# ----------------------------------------------------------------------------
# [E] Bias magnitudes
# ----------------------------------------------------------------------------
print()
print("=" * 142)
print("[E] Bias magnitudes — if cross-stream coupling moved into bias, biases will be large in absolute terms")
print("=" * 142)
print(f"{'layer':<32} {'parameter':<26} {'norm':>10} {'mean_abs':>10} {'max_abs':>10}")
print("-" * 142)
for name, module in model.named_modules():
    if not isinstance(getattr(module, 'integration_from_streams', None), torch.nn.ParameterList):
        continue
    ib = getattr(module, 'integrated_bias', None)
    if isinstance(ib, torch.nn.Parameter) and ib.numel() > 0:
        b = ib.detach().cpu()
        print(f"{name:<32} {'integrated_bias':<26} "
              f"{b.norm().item():>10.4f} {b.abs().mean().item():>10.4f} {b.abs().max().item():>10.4f}")
    sb = getattr(module, 'stream_biases', None)
    if isinstance(sb, torch.nn.ParameterList):
        for s_idx, p in enumerate(sb):
            b = p.detach().cpu()
            if b.numel() == 0:
                continue
            label = STREAM_LABELS.get(s_idx, f"s{s_idx}")
            print(f"{name:<32} {f'stream_biases[{label}]':<26} "
                  f"{b.norm().item():>10.4f} {b.abs().mean().item():>10.4f} {b.abs().max().item():>10.4f}")


# ----------------------------------------------------------------------------
# Aggregate verdict
# ----------------------------------------------------------------------------
mean_rel_dist = float(np.mean([r['rel_dist'] for r in A_rows]))
mean_cos_ones = float(np.mean([r['cos_ones'] for r in A_rows]))
mean_s2_ratio = float(np.nanmean([r['s2_ratio'] for r in A_rows]))
mean_resid_sr = float(np.nanmean([r['resid_sr'] for r in C_rows]))
mean_resid_rel = float(np.nanmean([r['rel_to_w'] for r in C_rows]))

ifs_sats = [r['sat'] for r in D_rows if r['component'] == 'integration_from_streams']
sw_sats = [r['sat'] for r in D_rows if r['component'] == 'stream_weights']
iw_sats = [r['sat'] for r in D_rows if r['component'] == 'integrated_weight']

print()
print("=" * 80)
print("AGGREGATE VERDICT")
print("=" * 80)
print(f"  integration_from_streams:")
print(f"    mean ||W - W_init|| / ||W_init||      = {mean_rel_dist:.4f}")
print(f"    mean cos(W, ones)                     = {mean_cos_ones:.4f}  (1.0 = exactly along ones)")
print(f"    mean s_2 / s_1                        = {mean_s2_ratio:.4f}  (1.0 = full rank, 0 = pure rank-1)")
print(f"    mean stable_rank(W - W_init)          = {mean_resid_sr:.2f}")
print(f"    mean ||resid|| / ||W||                = {mean_resid_rel:.4f}")
print(f"  cross-component saturation (mean):")
print(f"    stream_weights                        = {np.mean(sw_sats):.3f}  (Kaiming init expected ~0.4-0.7)")
if iw_sats:
    print(f"    integrated_weight                     = {np.mean(iw_sats):.3f}  (orthogonal init for square => 1.0)")
print(f"    integration_from_streams              = {np.mean(ifs_sats):.3f}  (constant init, suspect)")

print()
if mean_rel_dist < 0.05 and mean_cos_ones > 0.99:
    print("  >> integration_from_streams weights are ESSENTIALLY UNCHANGED FROM INIT.")
    print("     Optimizer never broke the symmetry. Training adapted around the fixed")
    print("     constant-1/N coupling via stream_weights, integrated_weight, and biases.")
elif mean_resid_sr < 1.5 and mean_resid_rel < 0.1:
    print("  >> integration_from_streams moved slightly along a SINGLE rank-1 direction;")
    print("     its full SVD is still dominated by the constant-init mode. Effective rank-1.")
elif mean_resid_sr > 1.5:
    print("  >> integration_from_streams residual has multi-dim structure but its NORM")
    print("     is small vs W_init. The constant mode dominates the SVD; trained structure")
    print("     is being masked. Renormalizing by subtracting the constant mode would expose it.")
else:
    print("  >> Mixed signal — inspect [A]/[C] rows directly to localize behavior.")


In [ ]:
import numpy as np
import torch

from src.models.linear_integration.li_net3 import li_resnet18


# Defensive STREAM_LABELS fallback (in case this cell runs without notebook scope).
if 'STREAM_LABELS' not in dir():
    STREAM_LABELS = {0: 'stream0', 1: 'stream1'}


# ----------------------------------------------------------------------------
# Build a freshly-initialized model with the SAME architecture.
#
# torch.manual_seed(42) makes the *statistical baseline* reproducible: Kaiming
# and orthogonal inits draw from fixed distributions, so any seed gives a
# fresh-init weight whose ||W_init|| is statistically equivalent. The seed
# choice does NOT need to match the original training run for this diagnostic
# to be valid — we only need a well-defined reference norm.
# Constant init (used for integration_from_streams) is deterministic regardless
# of RNG, so for THAT specific component the comparison is exact.
#
# device='cpu' avoids allocating GPU memory — we only read weights, no forward.
# use_amp=False is deliberate: autocast can change weight dtypes and would
# perturb the norm comparison; we want vanilla float32 for the reference.
# ----------------------------------------------------------------------------
torch.manual_seed(42)
fresh_model = li_resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    stream_input_channels=MODEL_CONFIG['stream_input_channels'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    dropout_p=MODEL_CONFIG['dropout_p'],
    device='cpu',
    use_amp=False,
)

trained_mods = dict(model.named_modules())
fresh_mods = dict(fresh_model.named_modules())

# Tolerant module-tree comparison. The trained model may have extra child
# modules attached during compile()/fit() (gpu_aug helpers, loss modules,
# cached buffers, etc.) that the fresh-init model doesn't have, and vice
# versa. We don't need exact equality — we just need the LIConv2d submodules
# (where integration_from_streams lives) to be present in both. Report the
# symmetric diff and proceed with the intersection.
t_mod_keys = set(trained_mods.keys())
f_mod_keys = set(fresh_mods.keys())
only_trained = t_mod_keys - f_mod_keys
only_fresh = f_mod_keys - t_mod_keys
common_mod_keys = t_mod_keys & f_mod_keys
if only_trained or only_fresh:
    print(f"INFO: module-tree differs between trained and fresh — using intersection ({len(common_mod_keys)} modules)")
    if only_trained:
        sample = sorted(only_trained)[:6]
        more = f" (+{len(only_trained) - 6} more)" if len(only_trained) > 6 else ""
        print(f"  only in trained model ({len(only_trained)}): {sample}{more}")
    if only_fresh:
        sample = sorted(only_fresh)[:6]
        more = f" (+{len(only_fresh) - 6} more)" if len(only_fresh) > 6 else ""
        print(f"  only in fresh model   ({len(only_fresh)}): {sample}{more}")
    print()

# Shape parity check on the parameter name intersection. Catches the failure
# mode where module names match but parameter shapes differ silently (e.g.
# MODEL_CONFIG has a different width_multiplier than the trained checkpoint).
t_named_params = dict(model.named_parameters())
f_named_params = dict(fresh_model.named_parameters())
common_param_keys = set(t_named_params.keys()) & set(f_named_params.keys())
for pname in common_param_keys:
    tp = t_named_params[pname]
    fp = f_named_params[pname]
    assert tp.shape == fp.shape, (
        f"Shape mismatch at {pname}: trained={tuple(tp.shape)} vs fresh={tuple(fp.shape)}. "
        f"MODEL_CONFIG doesn't match the trained checkpoint's architecture."
    )


# ----------------------------------------------------------------------------
# [F] Fresh-model comparison: ||DeltaW|| / ||W_init||
#
# Decision rules (per the diagnostic spec):
#   ratio < 0.05    -> weights barely moved (constant-init hypothesis CONFIRMED)
#   0.05 - 0.30     -> meaningful shift, still in init neighborhood
#   > 0.50          -> moved significantly; rank-1 structure is something the
#                      network LEARNED rather than something it stayed at.
#
# stream_weights[stem] (conv1) is included as an ANCHOR. The 7x7 stem stems are
# guaranteed to train under any reasonable optimizer; if their ratio is also
# tiny, the fresh-model construction itself is broken and the test is invalid.
# Expected stem anchor: ratio > 0.5.
# ----------------------------------------------------------------------------
print("=" * 142)
print("[F] Fresh-model comparison: ||W_trained - W_init|| / ||W_init||  (the deciding test)")
print("=" * 142)
print(f"{'layer':<32} {'component':<24} {'stream':<6} {'shape':<24} "
      f"{'||W_init||':>11} {'||W_trained||':>14} {'||DeltaW||':>11} {'ratio':>8}")
print("-" * 142)

agg: dict[str, list[float]] = {}

def _print_compare(layer, comp_name, stream_label, w_t, w_0, *, is_bias=False):
    init_norm = w_0.norm().item()
    trained_norm = w_t.norm().item()
    delta_norm = (w_t - w_0).norm().item()
    if is_bias:
        # Biases initialize uniform(-1/sqrt(fan_in), +1/sqrt(fan_in)) — small but
        # nonzero. Their ||init|| is small, so the ratio blows up to a number
        # that means "trained_norm / 0+" rather than "moved 1M× from init."
        # The actionable question for biases is "is this large enough to absorb
        # cross-stream signal?" — answered by the absolute trained norm.
        # Print '—' for the ratio and exclude from the agg verdict.
        print(f"{layer:<32} {comp_name:<24} {stream_label:<6} {str(tuple(w_t.shape)):<24} "
              f"{init_norm:>11.4f} {trained_norm:>14.4f} {delta_norm:>11.4f} {'—':>8}")
    else:
        ratio = delta_norm / max(init_norm, 1e-9)
        print(f"{layer:<32} {comp_name:<24} {stream_label:<6} {str(tuple(w_t.shape)):<24} "
              f"{init_norm:>11.4f} {trained_norm:>14.4f} {delta_norm:>11.4f} {ratio:>8.4f}")
        agg.setdefault(comp_name, []).append(ratio)


for name in common_mod_keys:
    t_mod = trained_mods[name]
    f_mod = fresh_mods[name]

    # integration_from_streams (the suspect — constant init, exact comparison)
    t_ifs = getattr(t_mod, 'integration_from_streams', None)
    f_ifs = getattr(f_mod, 'integration_from_streams', None)
    if isinstance(t_ifs, torch.nn.ParameterList) and isinstance(f_ifs, torch.nn.ParameterList):
        for s_idx in range(len(t_ifs)):
            w_t = t_ifs[s_idx].detach().cpu()
            w_0 = f_ifs[s_idx].detach().cpu()
            if w_t.numel() == 0:
                continue
            label = STREAM_LABELS.get(s_idx, f"s{s_idx}")
            _print_compare(name, 'integration_from_streams', label, w_t, w_0)

    # integrated_weight (W_prev / self-recurrence — orthogonal/kaiming init)
    t_iw = getattr(t_mod, 'integrated_weight', None)
    f_iw = getattr(f_mod, 'integrated_weight', None)
    if isinstance(t_iw, torch.nn.Parameter) and isinstance(f_iw, torch.nn.Parameter):
        w_t = t_iw.detach().cpu()
        w_0 = f_iw.detach().cpu()
        if w_t.numel() > 0:
            _print_compare(name, 'integrated_weight', '-', w_t, w_0)

    # integrated_bias (small param but tracks "is the bias absorbing signal?")
    t_ib = getattr(t_mod, 'integrated_bias', None)
    f_ib = getattr(f_mod, 'integrated_bias', None)
    if isinstance(t_ib, torch.nn.Parameter) and isinstance(f_ib, torch.nn.Parameter):
        b_t = t_ib.detach().cpu()
        b_0 = f_ib.detach().cpu()
        if b_t.numel() > 0:
            _print_compare(name, 'integrated_bias', '-', b_t, b_0, is_bias=True)

    # ANCHOR: conv1 stream_weights — 7x7 stems must train.
    if name == 'conv1':
        t_sw = getattr(t_mod, 'stream_weights', None)
        f_sw = getattr(f_mod, 'stream_weights', None)
        if isinstance(t_sw, torch.nn.ParameterList) and isinstance(f_sw, torch.nn.ParameterList):
            for s_idx in range(len(t_sw)):
                w_t = t_sw[s_idx].detach().cpu()
                w_0 = f_sw[s_idx].detach().cpu()
                if w_t.numel() == 0:
                    continue
                label = STREAM_LABELS.get(s_idx, f"s{s_idx}")
                _print_compare(name, 'stream_weights[stem]*', label, w_t, w_0)


# ----------------------------------------------------------------------------
# Aggregate verdict
# ----------------------------------------------------------------------------
print()
print("=" * 80)
print("AGGREGATE: mean ||DeltaW|| / ||W_init|| per component")
print("=" * 80)
for comp, ratios in agg.items():
    mean_r = float(np.mean(ratios))
    if comp == 'stream_weights[stem]*':
        if mean_r < 0.5:
            verdict = "ANCHOR FAIL — fresh-model construction broken or training never happened"
        else:
            verdict = "anchor OK — confirms fresh-model construction is valid"
    elif mean_r < 0.05:
        verdict = "ESSENTIALLY UNCHANGED FROM INIT (constant-init hypothesis confirmed)"
    elif mean_r < 0.30:
        verdict = "shifted but still in init neighborhood"
    elif mean_r > 0.50:
        verdict = "moved significantly; not at init"
    else:
        verdict = "intermediate (0.30 - 0.50)"
    print(f"  {comp:<28} n={len(ratios):3d}   mean = {mean_r:.4f}    {verdict}")


# ----------------------------------------------------------------------------
# Final story
# ----------------------------------------------------------------------------
print()
print("=" * 80)
print("Conclusion")
print("=" * 80)
ifs_mean = float(np.mean(agg.get('integration_from_streams', [float('nan')])))
iw_mean = float(np.mean(agg.get('integrated_weight', [float('nan')])))
stem_mean = float(np.mean(agg.get('stream_weights[stem]*', [float('nan')])))

if not np.isnan(stem_mean) and stem_mean < 0.5:
    print("  Cannot conclude — anchor failed. Fix fresh-model construction first.")
elif ifs_mean < 0.05 and iw_mean > 0.30:
    print("  CONFIRMED: integration_from_streams is frozen at constant init while")
    print("  integrated_weight (W_prev) trained normally. Cross-stream coupling")
    print("  happens through W_prev's self-recurrence and integrated_bias, NOT")
    print("  through the per-stream 1x1 W_int_i matrices. The constant-1/N init")
    print("  is a saddle point the optimizer never escaped.")
elif ifs_mean < 0.05 and iw_mean < 0.30:
    print("  integration_from_streams AND integrated_weight both barely moved.")
    print("  Either a deeper optimization problem (LR too low, gradients vanishing)")
    print("  or training stopped before the integration mechanism began updating.")
elif 0.05 <= ifs_mean <= 0.30 and iw_mean > 0.30:
    print("  PARTIAL: integration_from_streams shifted slightly from constant init")
    print("  (~5-30% relative drift) while integrated_weight trained normally.")
    print("  The constant mode still dominates the SVD; the small residual structure")
    print("  is what training added. Check cell 36 [C]: stable_rank(residual) tells")
    print("  you whether the trained perturbation is itself rank-1 (collapsed) or")
    print("  multi-dim (real but small). Cell 36 [B] product column tells you")
    print("  whether the dominant SV is the constant mode or a learned direction.")
elif ifs_mean > 0.30:
    print("  integration_from_streams moved meaningfully — the rank-1 structure")
    print("  is something the network LEARNED, not something it stayed at.")
    print("  Investigate: top SV vector cosine with ones (cell 36 [B]) tells you")
    print("  whether the learned rank-1 mode is the constant-mode or a different one.")
else:
    # Falls through when 0.05 <= ifs_mean <= 0.30 AND iw_mean <= 0.30 — partial
    # drift in integration_from_streams but the integrated stream's self-conv
    # also barely moved. Less common; symptoms ambiguous.
    print("  Mixed signal: integration_from_streams drifted slightly but")
    print("  integrated_weight also barely moved. Check stem anchor and per-layer")
    print("  rows — possibly an early-stopped run or an optimization slowdown")
    print("  affecting late layers more than early ones.")


## 14. Training Curves + Gradient Health + Stream Loss Decomposition

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Row 1: Standard training curves (restored from original + adapted for no val set) ---

# Loss curve
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curve with per-stream curves
axes[0, 1].plot([acc*100 for acc in history['train_accuracy']], label='Full Model Train', linewidth=2, color='green')
if 'train_mca' in history and history['train_mca']:
    axes[0, 1].plot([m*100 for m in history['train_mca']], label='Train MCA', linewidth=2, color='darkorange', linestyle=':')

# Add per-stream curves (always available with stream_monitoring=True)
stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(stream_train_colors)
    axes[0, 1].plot([acc*100 for acc in history[f'stream_{i}_train_acc']],
                label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
                color=stream_train_colors[color_idx])

axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_yticks([20, 40, 60, 80, 100])
axes[0, 1].set_title('Training Accuracy\n(Full Model = Integrated Stream)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='lower right')
# Draw gridlines manually: alpha=0.3 for multiples of 10, alpha=0.2 for 5, 15, 25...
for y in range(0, 101, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.3, linewidth=0.5)
for y in range(5, 100, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.2, linewidth=0.5)
axes[0, 1].grid(True, axis='x', alpha=0.3)

# Learning rate curve with per-stream LRs
sampled_lrs = history['learning_rates'][::max(1, len(history['learning_rates'])//100)]
axes[0, 2].plot(sampled_lrs, linewidth=2, color='green', label='Base LR')

# Add per-stream LRs (always available with stream_monitoring=True)
lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(lr_colors)
    axes[0, 2].plot(history[f'stream_{i}_lr'], linewidth=1, alpha=0.7, linestyle='--',
                color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} LR')
    # Plot stem LR if stem_lr_multiplier was active (separate higher LR for conv1)
    if f'stem_{i}_lr' in history:
        axes[0, 2].plot(history[f'stem_{i}_lr'], linewidth=1.5, alpha=0.5, linestyle=':',
                    color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} Stem LR')

axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Learning Rate', fontsize=12)
axes[0, 2].set_yscale('log')
axes[0, 2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=9, loc='upper right')
axes[0, 2].grid(True, alpha=0.3)

# --- Row 2: New diagnostics ---

# Gradient norms over epochs (values are dicts with mean/max/min)
if 'gradient_norms' in history and history['gradient_norms']:
    grad_epochs = range(len(history['gradient_norms']))
    for i in range(len(MODEL_CONFIG['stream_input_channels'])):
        key = f'stream_{i}'
        norms = [d.get(key, {}).get('mean', 0) for d in history['gradient_norms']]
        color = stream_val_colors[i % len(stream_val_colors)]
        axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5)
    shared_norms = [d.get('shared', {}).get('mean', 0) for d in history['gradient_norms']]
    axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Gradient Norm (pre-clip, log)', fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms (mean)', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No gradient data\n(gradient_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 0].transAxes, fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms', fontsize=14, fontweight='bold')

contrib_keys = [f'stream_{i}_train_acc' for i in range(len(MODEL_CONFIG['stream_input_channels']))]
if contrib_keys[0] in history:
    import math
    n_streams = len(MODEL_CONFIG['stream_input_channels'])
    baseline_vals = history['train_accuracy']
    for i in range(n_streams):
        color = stream_val_colors[i % len(stream_val_colors)]
        other = (i + 1) % n_streams if n_streams == 2 else i
        other_vals = history[f'stream_{other}_train_acc']
        contrib = []
        epochs_eval = []
        for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
            if not math.isnan(other_acc):
                contrib.append((base - other_acc) * 100)
                epochs_eval.append(e)
        axes[1, 1].plot(epochs_eval, contrib,
                       label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5, marker='o', markersize=3)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Contribution (pp)', fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution\n(Baseline − Acc w/o Stream)', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No stream data\n(stream_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution', fontsize=14, fontweight='bold')




# Gradient health status summary
if 'gradient_health' in history and history['gradient_health']:
    axes[1, 2].axis('off')
    health_text = "Gradient Health Summary:\n\n"
    status_counts = {}
    for h in history['gradient_health']:
        status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        health_text += f"  {status}: {count} epochs\n"
    axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')
else:
    axes[1, 2].text(0.5, 0.5, 'No gradient health data\n(gradient_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 2].transAxes, fontsize=12)
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{checkpoint_dir}/training_diagnostics.pdf", dpi=150, bbox_inches='tight')
plt.show()

print(f"Training diagnostics saved to: {checkpoint_dir}/training_diagnostics.pdf")

## 15. Integration Weight Evolution During Training

How did the learned integration priorities change over training? Did the model start RGB-heavy and shift toward HHA?

In [ ]:
# Integration weight evolution visualization
evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# Stream backbone weight norm evolution (RGB, HHA, Integrated)
if 'stream_weight_norms' in history:
    evo_viz.plot_stream_weight_norms(history, save_path=f"{checkpoint_dir}/stream_weight_evolution.pdf")
    print(f"Stream weight norm evolution saved.")
else:
    print("No stream weight norm data found in history.")

In [ ]:
# Plot norm evolution from training history
if 'integration_weight_norms' in history:
    evo_viz.plot_norm_evolution(history, save_path=f"{checkpoint_dir}/integration_weight_evolution.pdf")
    print(f"Integration weight norm evolution saved.")
else:
    print("No integration weight norm data found in history.")

In [ ]:
# Plot full weight snapshots if saved
snapshot_dir = TRAIN_CONFIG.get('integration_snapshot_path')
if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
    # Full grid as PNG (all layers, raster — too heavy for PDF)
    evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{checkpoint_dir}/integration_weight_snapshots.png")
    print(f"Integration weight snapshot heatmaps saved (full, PNG).")

    # Early layers only as PDF (vector, paper-ready)
    for layer_name in ['conv1', 'layer1']:
        evo_viz.plot_snapshot_heatmaps(
            snapshot_dir,
            layer_filter=layer_name,
            save_path=f"{checkpoint_dir}/integration_weight_snapshots_{layer_name}.pdf"
        )
    print(f"Integration weight snapshot heatmaps saved (conv1 + layer1, PDF).")
else:
    print("No integration weight snapshots found.")

In [ ]:
# Visualize learned first-layer conv filters (7x7 kernels)
iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)
iw_viz.visualize_conv1_filters(save_path=f'{checkpoint_dir}/conv1_filters.pdf')
print('Conv1 filter visualization saved.')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import torch

from src.data_utils.sunrgbd_dataset import SUNRGBDDataset

# Build a viz-only dataset with normalize=False so HHA values are in
# original units (1/m, meters, degrees). Augmentation is implicitly off
# for split='test'.
viz_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='test',
    crop_size=224,
    normalize=False,
    use_hha=True,
)

NUM_VIZ_SAMPLES = 6
indices = list(range(0, len(viz_dataset), len(viz_dataset) // NUM_VIZ_SAMPLES))[:NUM_VIZ_SAMPLES]

fig, axes = plt.subplots(NUM_VIZ_SAMPLES, 4, figsize=(13, 3.0 * NUM_VIZ_SAMPLES))
for row, idx in enumerate(indices):
    rgb, hha, label = viz_dataset[idx]
    rgb_np = rgb.permute(1, 2, 0).numpy()  # [H, W, 3] float32 in [0, 1]
    hha_np = hha.numpy()                   # [3, H, W] float32 (NaN-replaced)

    cls = viz_dataset.CLASS_NAMES[label]
    axes[row, 0].imshow(rgb_np)
    axes[row, 0].set_title(f"#{idx}  {cls}", fontsize=10)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(hha_np[0], cmap="viridis", vmin=0.0, vmax=3.0)
    axes[row, 1].set_title("HHA[0] disparity (1/m)\n[bright=near]", fontsize=8)
    axes[row, 1].axis("off")

    h = hha_np[1]
    h_abs = max(np.percentile(np.abs(h), 99), 1e-3)
    axes[row, 2].imshow(h, cmap="RdBu_r", vmin=-h_abs, vmax=h_abs)
    axes[row, 2].set_title("HHA[1] height (m)\n[blue=below floor, red=above]", fontsize=8)
    axes[row, 2].axis("off")

    norm = TwoSlopeNorm(vmin=0.0, vcenter=90.0, vmax=180.0)
    axes[row, 3].imshow(hha_np[2], cmap="RdBu_r", norm=norm)
    axes[row, 3].set_title("HHA[2] angle vs gravity (deg)\n[blue=ceiling, white=wall, red=floor]", fontsize=8)
    axes[row, 3].axis("off")

fig.suptitle("HHA channel verification (post-training, test split)", fontsize=12, y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.985))
out_path = f"{checkpoint_dir}/hha_channel_verification.png"
plt.savefig(out_path, dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved HHA verification grid to: {out_path}")
print("Floors should read clearly red and walls white in the angle column.")


## 16. Save Results & Model

In [ ]:
import json
import torch

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{checkpoint_dir}/training_history.json"
with open(history_path, 'w') as f:
    # Build pathway analysis dict with all returned data
    pa_json = {
        'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
        'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
        'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
        'samples_analyzed': pathway_analysis['samples_analyzed'],
    }

    json_history = {
        'train_loss': [float(x) for x in history['train_loss']],
        'train_accuracy': [float(x) for x in history['train_accuracy']],
        'learning_rates': [float(x) for x in history['learning_rates']],
        # Per-stream and stem LR curves (when stream_monitoring=True)
        **{f'stream_{i}_lr': [float(x) for x in history.get(f'stream_{i}_lr', [])]
           for i in range(2)},
        **{f'stem_{i}_lr': [float(x) for x in history.get(f'stem_{i}_lr', [])]
           for i in range(2) if f'stem_{i}_lr' in history},
        'model_config': MODEL_CONFIG,
        'dataset_config': DATASET_CONFIG,
        'augmentation_config': AUGMENTATION_CONFIG.to_dict(),
        'stream_specific_config': STREAM_SPECIFIC_CONFIG,
        'scheduler_config': SCHEDULER_CONFIG,
        'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v for k, v in TRAIN_CONFIG.items()},
        'train_mca': [float(x) for x in history.get('train_mca', [])],
        'test_results': {
            'loss': float(results['loss']),
            'accuracy': float(results['accuracy']),
            'mean_class_accuracy': float(results.get('mean_class_accuracy', 0)),
        },
        'pathway_analysis': pa_json,
    }

    json.dump(json_history, f, indent=2)

print(f"Training history saved: {history_path}")

# Save final model
final_model_path = f"{checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': MODEL_CONFIG,
    'history': history,
    'test_accuracy': results['accuracy']
}, final_model_path)

print(f"Final model saved: {final_model_path}")

# List saved files
print(f"\nAll results saved to: {checkpoint_dir}")
!ls -lh {checkpoint_dir}

print("\n" + "=" * 60)

## 17. Internal CNN Visualization Suite

Everything below runs on the **trained model** with the **test set**. Each cell is independent — run whichever analyses interest you.

In [ ]:
# --- 17a. Feature Map Visualization ---
# "What does the CNN see at each layer?"
# Three modes: full model, single-stream isolated, ablation
# Compare layer1 (early/texture) vs layer4 (late/semantic)

fm_viz = FeatureMapVisualizer(model, stream_labels=STREAM_LABELS)

# Get a single test sample
test_iter = iter(test_loader)
sample_batch = next(test_iter)
*stream_batches, labels = sample_batch
# Take first sample
stream_inputs = [s[0:1].to(model.device) for s in stream_batches]

print(f"Sample label: {labels[0].item()} ({class_names[labels[0].item()] if 'class_names' in dir() else '?'})")

for layer in ['layer1', 'layer4']:
    print(f"\n{'='*60}")
    print(f"  {layer.upper()} FEATURE MAPS")
    print(f"{'='*60}")

    # Mode 1: Full model view (all streams + integrated)
    print(f"\n--- Full Model View ({layer}) ---")
    fm_viz.visualize(stream_inputs, layer=layer, top_k=8,
                     save_path=f"{checkpoint_dir}/featuremaps_full_{layer}.png")

    # Mode 2: Per-stream isolated views
    for i, label in STREAM_LABELS.items():
        print(f"\n--- {label} Stream Isolated View ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='stream', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_{label.lower()}_{layer}.png")

    # Mode 3: Ablation (what happens when we remove a stream?)
    for i, label in STREAM_LABELS.items():
        print(f"\n--- Ablation: {label} Blanked ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='ablation', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_ablation_{label.lower()}_{layer}.png")

# Batch-averaged feature maps at both layers
for layer in ['layer1', 'layer4']:
    print(f"\n--- Batch-Averaged Feature Maps ({layer}, 32 samples) ---")
    fm_viz.visualize_batch(test_loader, layer=layer, n=32, top_k=8,
                           save_path=f"{checkpoint_dir}/featuremaps_batch_avg_{layer}.png")

print("\nFeature map visualizations complete!")

In [ ]:
# --- 17b. Stream Contribution Decomposition ---
# THE unique LINet3 visualization: how much does each stream contribute
# to each neuron's activation in the integrated pathway?

contrib_viz = StreamContributionVisualizer(model, stream_labels=STREAM_LABELS)

# Single image contribution at layer4
print("--- Stream Contributions (layer4, single sample) ---")
contrib_viz.visualize(stream_inputs, layer='layer4',
                      save_path=f"{checkpoint_dir}/contributions_layer4.pdf")

# Batch-averaged contributions (more representative)
print("\n--- Batch-Averaged Contributions (layer4, 32 samples) ---")
contrib_viz.visualize_batch(test_loader, layer='layer4', n=32,
                            save_path=f"{checkpoint_dir}/contributions_batch_layer4.pdf")

# Multi-layer comparison
for layer in ['layer1', 'layer2', 'layer3', 'layer4']:
    print(f"\n--- Contributions at {layer} ---")
    contrib_viz.visualize(stream_inputs, layer=layer,
                          save_path=f"{checkpoint_dir}/contributions_{layer}.pdf")

print("\nStream contribution decomposition complete!")

In [ ]:
# --- 17c. Stream-Decomposed Grad-CAM ---
# Where does each stream focus its attention?

gradcam = StreamGradCAM(model, stream_labels=STREAM_LABELS)

# Integrated Grad-CAM (standard: where does the full model look?)
print("--- Integrated Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='integrated',
                  save_path=f"{checkpoint_dir}/gradcam_integrated_layer4.png")

# Per-stream isolated Grad-CAM (where does each stream look independently?)
for i, label in STREAM_LABELS.items():
    print(f"\n--- {label} Stream Grad-CAM (layer4) ---")
    gradcam.visualize(stream_inputs, layer='layer4', mode='stream', stream_idx=i,
                      save_path=f"{checkpoint_dir}/gradcam_{label.lower()}_layer4.png")

# Decomposed mode: contribution maps weighted by Grad-CAM importance
print("\n--- Decomposed Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='decomposed',
                  save_path=f"{checkpoint_dir}/gradcam_decomposed_layer4.png")

# Multi-layer Grad-CAM (early=texture, late=semantics)
for layer in ['layer2', 'layer3', 'layer4']:
    print(f"\n--- Integrated Grad-CAM at {layer} ---")
    gradcam.visualize(stream_inputs, layer=layer, mode='integrated',
                      save_path=f"{checkpoint_dir}/gradcam_integrated_{layer}.png")

print("\nGrad-CAM visualizations complete!")

In [ ]:
# --- 17d. Integration Weight Visualization ---
# Visualize the learned integration_from_streams weights per layer

iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)

# Weight heatmaps per layer and stream
print('--- Integration Weights (Heatmaps) ---')
iw_viz.visualize_weights(save_path=f'{checkpoint_dir}/integration_weights.png')

# Cross-stream comparison (relative weight magnitudes per layer)
print('\n--- Cross-Stream Weight Comparison ---')
iw_viz.visualize_cross_stream(save_path=f'{checkpoint_dir}/integration_cross_stream.pdf')

# Effective rank via SVD (how low-dimensional is the integration?)
print('\n--- Effective Rank (SVD) ---')
ranks = iw_viz.compute_effective_rank()
for layer, r in ranks.items():
    print(f'  {layer}: {[f"{x:.1f}" for x in r]}')

print('\nIntegration weight visualization complete!')

In [ ]:
# --- 17e. Stream Redundancy Analysis ---
# Are RGB and HHA learning the same features? Or complementary ones?
# Uses centered cosine similarity between stream feature maps at each layer.

redundancy = StreamRedundancyAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Stream Redundancy (Centered Cosine Similarity) ---')
sim_results = redundancy.analyze(
    test_loader,
    n=128,  # Average over 128 samples
    save_path=f'{checkpoint_dir}/stream_redundancy.pdf'
)

# Print similarity matrices
for layer_name, sim_matrix in sim_results.items():
    print(f'\n{layer_name}:')
    for i in range(sim_matrix.shape[0]):
        row = '  '.join(f'{sim_matrix[i,j]:.3f}' for j in range(sim_matrix.shape[1]))
        print(f'  {STREAM_LABELS.get(i, f"S{i}")}: {row}')

print('\nStream redundancy analysis complete!')

In [ ]:
# --- 17f. Per-Class Stream Dominance ---
# Which scenes rely on RGB vs HHA?
# "HHA matters more for bathrooms, RGB dominates corridors"

# Build class name mapping
class_name_map = {i: name for i, name in enumerate(class_names)} if 'class_names' in dir() else None

dominance = PerClassDominanceAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Per-Class Stream Dominance (layer4) ---')
class_dominance = dominance.analyze(
    test_loader,
    layer='layer4',
    class_names=class_name_map,
    save_path=f'{checkpoint_dir}/per_class_dominance.pdf'
)

# Print per-class ratios
print('\nPer-class stream contribution ratios:')
for cls_idx, ratios in sorted(class_dominance.items()):
    name = class_name_map[cls_idx] if class_name_map else f'Class {cls_idx}'
    ratio_str = ', '.join(f'{STREAM_LABELS.get(i, f"S{i}")}: {r:.2%}' for i, r in enumerate(ratios))
    print(f'  {name}: {ratio_str}')

print('\nPer-class dominance analysis complete!')

In [ ]:
# --- 17g. Misclassification Analysis + Sample Comparison ---
# Find misclassified samples and compare with correctly classified ones

print('--- Finding Misclassified Samples ---')
misclassified = find_misclassified(model, test_loader, n=10)

print(f'Found {len(misclassified)} misclassified samples:')
for i, mc in enumerate(misclassified[:5]):
    true_name = class_names[mc['true_label']] if 'class_names' in dir() else str(mc['true_label'])
    pred_name = class_names[mc['predicted_label']] if 'class_names' in dir() else str(mc['predicted_label'])
    print(f'  [{i}] True: {true_name}, Predicted: {pred_name}, Confidence: {mc["confidence"]:.2%}')

# Grad-CAM on first misclassified sample
if misclassified:
    mc_sample = misclassified[0]
    mc_inputs = [s.to(model.device) for s in mc_sample['stream_inputs']]
    true_name = class_names[mc_sample['true_label']] if 'class_names' in dir() else str(mc_sample['true_label'])
    pred_name = class_names[mc_sample['predicted_label']] if 'class_names' in dir() else str(mc_sample['predicted_label'])
    print(f'\n--- Grad-CAM on Misclassified: True={true_name}, Pred={pred_name} ---')
    gradcam.visualize(mc_inputs, layer='layer4', mode='decomposed',
                      save_path=f'{checkpoint_dir}/gradcam_misclassified_0.png')

# Compare correct vs misclassified from same class
if misclassified:
    target_class = misclassified[0]['true_label']
    print(f'\n--- Finding correctly classified sample from class {class_names[target_class] if "class_names" in dir() else target_class} ---')

    # Find a correctly classified sample from the same class
    correct_sample = None
    model.eval()
    with torch.no_grad():
        for batch_data in test_loader:
            *stream_batches, targets = batch_data
            stream_batches_dev = [s.to(model.device) for s in stream_batches]
            targets_dev = targets.to(model.device)
            logits = model(stream_batches_dev)
            preds = logits.argmax(dim=1)
            # Find correctly classified samples of the target class
            mask = (targets_dev == target_class) & (preds == target_class)
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0][0].item()
                correct_sample = {
                    'stream_inputs': [s[idx:idx+1].cpu() for s in stream_batches],
                    'true_label': target_class,
                    'predicted_label': target_class,
                    'confidence': torch.softmax(logits[idx], dim=0)[target_class].item(),
                }
                break

    if correct_sample is not None:
        print(f'  Found correct sample (confidence: {correct_sample["confidence"]:.2%})')
        print('\n--- Correct vs Misclassified Comparison ---')
        compare_samples(
            model,
            correct_sample=correct_sample,
            misclassified_sample=misclassified[0],
            layer='layer4',
            stream_labels=STREAM_LABELS,
            save_path=f'{checkpoint_dir}/compare_samples.png'
        )
    else:
        print('  No correctly classified sample found for this class.')

print('\nMisclassification analysis complete!')

In [ ]:
# --- 17h. Train vs Test Activation Divergence ---
# Does the model see different activation distributions on train vs test?
# Uses MMD (Maximum Mean Discrepancy) per layer.

div_analyzer = ActivationDivergenceAnalyzer(model)

print('--- Train vs Test Activation Divergence ---')
divergence = div_analyzer.analyze(
    train_loader,
    test_loader,
    n=128,
    save_path=f'{checkpoint_dir}/activation_divergence.pdf'
)

for layer_name, metrics in divergence.items():
    print(f'  {layer_name}: MMD={metrics["mmd"]:.4f}')

print('\nActivation divergence analysis complete!')

In [ ]:
# --- 17i. BN Stats Reset Experiment (Oracle Diagnostic) ---
# WARNING: This is a DIAGNOSTIC tool, not a deployable fix.
# It recomputes BN running stats on test data (oracle) to check if
# BN statistics drift causes the generalization gap.

import copy

# Save original accuracy
original_test_results = model.evaluate(test_loader)
original_acc = original_test_results['accuracy']
print(f'Original test accuracy: {original_acc*100:.2f}%')

# Control: recompute BN stats on TRAIN set (should be ~same)
print('\n--- Control: Recompute BN stats on TRAIN set ---')
model_control = copy.deepcopy(model)
reset_bn_stats(model_control, train_loader)
control_results = model_control.evaluate(test_loader)
control_acc = control_results['accuracy']
print(f'After train BN reset: {control_acc*100:.2f}% (delta: {(control_acc-original_acc)*100:+.2f}%)')

# Oracle: recompute BN stats on TEST set
print('\n--- Oracle: Recompute BN stats on TEST set ---')
model_oracle = copy.deepcopy(model)
reset_bn_stats(model_oracle, test_loader)
oracle_results = model_oracle.evaluate(test_loader)
oracle_acc = oracle_results['accuracy']
print(f'After test BN reset (oracle): {oracle_acc*100:.2f}% (delta: {(oracle_acc-original_acc)*100:+.2f}%)')

# Interpretation
print('\n--- Interpretation ---')
oracle_delta = (oracle_acc - original_acc) * 100
if abs(oracle_delta) > 2:
    print(f'BN stats drift accounts for ~{oracle_delta:+.1f}% of the gap.')
    print('Consider: test-time BN adaptation, larger batch size, or more training data.')
else:
    print(f'BN stats drift is minimal ({oracle_delta:+.1f}%). Gap is likely from other sources.')

del model_control, model_oracle  # Free memory
print('\nBN reset experiment complete!')

## 18. Summary

All training diagnostics and visualization analyses are saved to the checkpoint directory on Google Drive.

**Saved models:**
- `best_model.pt` - Best model checkpoint (by training loss, since no val set)
- `final_model.pt` - Final model with full state dict, optimizer, scheduler, history

**Saved data:**
- `training_history.json` - Full training history, configs, test results, pathway analysis
- `integration_snapshots/` - Periodic full integration weight snapshots (every N epochs)

**Saved visualizations:**
- `training_diagnostics.png` - 2x3 grid: loss, accuracy, LR, gradient norms, stream losses, gradient health
- `integration_weight_evolution.png` - Per-stream integration weight norms over training epochs
- `integration_weight_snapshots.png` - Detailed weight heatmaps at snapshot epochs
- `featuremaps_*.png` - What the CNN sees (full model, per-stream isolated, ablation, batch-averaged)
- `contributions_*.png` - Per-stream contribution magnitudes at each layer
- `gradcam_*.png` - Spatial attention maps (integrated, per-stream, decomposed, multi-layer)
- `gradcam_misclassified_0.png` - Decomposed Grad-CAM on a misclassified sample
- `integration_weights.png` - Learned fusion weight heatmaps per layer
- `integration_cross_stream.png` - Cross-stream weight magnitude comparison
- `stream_redundancy.png` - Centered cosine similarity between stream features per layer
- `per_class_dominance.png` - Which scenes rely on RGB vs HHA
- `activation_divergence.png` - Train vs test activation distribution shift (MMD) per layer
- `compare_samples.png` - Correct vs misclassified side-by-side (Grad-CAM + contributions)